# PV-UAD → AG-ReID.v2 Aerial→Ground
## E2=60 → E3=25 + trainable TransReID initialization + flip-TTA

Bản này chuyển pipeline WHU-MARS-1000 đã đạt **11.877 mAP** sang
**AG-ReID.v2** tại:

`/kaggle/input/datasets/thienbao1604/ag-reid-v2/AG-ReID.v2/AG-ReID.v2.`

Protocol mặc định: **aerial → ground**.

Những phần giữ từ pipeline 11.877:

- ViT-Base 768-D, stride `[16,16]`;
- pretrained MSMT17 TransReID → transplant vào UAD backbone và **fine-tune toàn bộ**;
- E2 = **60 epoch**, SGD, LR 0.008, global batch 24;
- VF-ProCA + GPD;
- E3 = **25 epoch**, LR 1e-4, head LR ×5;
- E2 teacher feature-preservation + scenario-hard GPD;
- final horizontal-flip TTA;
- Kaggle T4×2 multi-session resume.

Thay đổi bắt buộc vì AG-ReID.v2 là **RGB-only**:

- bỏ TIR distillation (không có NIR/TIR để distill);
- dùng **view-balanced PKM**: mỗi identity ưu tiên 2 ground + 2 aerial images trong group 4;
- remap camera để view-factorized ProCA hiểu đúng:
  `C2 wearable→0`, `C3 CCTV→1`, `C0 UAV→2`, với `GROUND_MAX_CAMID=1`;
- evaluator chạy đúng query aerial C0 → gallery ground C2/C3.

Ngoài ra notebook **không giả định best checkpoint = last checkpoint**:
E3 được warm-start từ **E2 epoch 60 + prototype cùng epoch 60**, và cuối cùng
report cả **best-E3 + TTA** lẫn **E3 epoch 25 + TTA** để bạn nhìn trực tiếp hai checkpoint có thật sự giống nhau hay không.

> **v6 DDP evaluation fix:** AG-ReID.v2 has only one modality. In the previous
> version, distributed evaluation assigned that single validation loader to
> rank 0 only, while rank 1 entered a NCCL synchronization. That can trigger
> the ~10-minute ProcessGroupNCCL watchdog timeout around an evaluation epoch.
> This version makes both GPUs evaluate the single AG-ReID.v2 loader
> independently with the **unwrapped model** (no DDP collectives during
> inference), then synchronizes only after both ranks have finished.
>
> The internal `PIPELINE_TAG` intentionally remains the v5 tag so an attached
> v5 output containing `pvuad_last_checkpoint.pth` can be resumed instead of
> throwing away the previous 2.5 hours.

In [1]:
# ============================================================
# 0. SETTINGS — AG-ReID.v2 aerial->ground, multi-session
# ============================================================

import time as _time
NOTEBOOK_SESSION_STARTED_UNIX = _time.time()

PIPELINE_NAME = "PVUAD_AGREIDV2_E2_E3_TRANSREID"
PIPELINE_TAG = "agreidv2_a2g_e2_60_e3_25_transreid768_b24_v5_manifestfix"

# Exact Kaggle dataset path supplied by the user.
AGREIDV2_INPUT = "/kaggle/input/datasets/thienbao1604/ag-reid-v2/AG-ReID.v2/AG-ReID.v2."
AG_PROTOCOL = "aerial_to_ground"

SOURCE_DIR_OVERRIDE = None

# Official MSMT17 full TransReID checkpoint.
TRANSREID_CHECKPOINT_OVERRIDE = None
PRETRAIN_PATH_OVERRIDE = None
RUN_TRANSREID_INIT_SMOKE_TEST = True

# Keep the winning WHU-1000 backbone recipe.
TRANSREID_TARGET_STRIDE = [16, 16]
TRANSREID_TARGET_EMBED_DIM = 768
USE_SOURCE_SIE = False
USE_SOURCE_JPM_LOCAL_BRANCHES = False

# Optional emergency overrides. Normally leave all three as None.
E2_RESUME_PATH_OVERRIDE = None
E2_COMPLETED_DIR_OVERRIDE = None
E3_RESUME_PATH_OVERRIDE = None

AUTO_RESUME = True
ALLOW_LEGACY_E2_BOOTSTRAP = False
RUN_TRAINING = True
RUN_FINAL_FLIP_TTA = True
RUN_METADATA_PAIRING_CHECK = True

# Keep the stage boundary explicit so a Kaggle 12h session can be resumed safely.
ALWAYS_START_E3_IN_NEW_SESSION = True

# Fixed horizons requested by the user.
E2_EPOCHS = 60
E2_GLOBAL_BATCH = 24
E2_BASE_LR = 0.008

E3_EPOCHS = 25
E3_GLOBAL_BATCH = 24
E3_BASE_LR = 1e-4
E3_HEAD_LR_MULTIPLIER = 5.0

# E3 keeps the transferable pieces of the WHU winner.
# TIR distillation is intentionally disabled because AG-ReID.v2 is RGB-only.
FEATURE_PRESERVE_WEIGHT = 0.05
SCENARIO_HARD_WEIGHT = 0.15
SCENARIO_HARD_TOPK = 5
SCENARIO_HARD_MARGIN = 0.05
SCENARIO_HARD_TAU = 0.05
SCENARIO_HARD_START_EPOCH = 3

# AG-ReID.v2 camera remap used by the patched loader:
# C2 wearable -> 0 (ground), C3 CCTV -> 1 (ground), C0 UAV -> 2 (aerial).
GROUND_MAX_CAMID = 1

# One Kaggle session lasts at most 12 h.
SESSION_HARD_STOP_HOURS = 10.5
SESSION_STOP_RESERVE_MINUTES = 15
CHECKPOINT_EVERY_MINUTES = 30
TIME_CHECK_EVERY_STEPS = 10
MIN_EVAL_REMAINING_MINUTES = 25
MIN_E3_START_REMAINING_MINUTES = 75
MIN_TTA_START_REMAINING_MINUTES = 35

NUM_GPUS = 2
NUM_WORKERS = 4
RANDOM_SEED = 1234

if AG_PROTOCOL != "aerial_to_ground":
    raise ValueError("This notebook is locked to AG-ReID.v2 aerial->ground.")
if E2_GLOBAL_BATCH != 24 or abs(E2_BASE_LR - 0.008) > 1e-12:
    raise ValueError("E2 must stay at global batch 24 / LR 0.008.")
if E3_GLOBAL_BATCH != 24 or abs(E3_BASE_LR - 1e-4) > 1e-12:
    raise ValueError("E3 must stay at global batch 24 / LR 1e-4.")
if TRANSREID_TARGET_STRIDE != [16, 16]:
    raise ValueError("Keep UAD stride [16,16] for this transfer experiment.")
if TRANSREID_TARGET_EMBED_DIM != 768:
    raise ValueError("This pipeline must keep the UAD 768-D feature space.")
if USE_SOURCE_SIE or USE_SOURCE_JPM_LOCAL_BRANCHES:
    raise ValueError("Source SIE/JPM local branches must remain disabled.")

SESSION_DEADLINE_UNIX = (
    NOTEBOOK_SESSION_STARTED_UNIX + 3600.0 * SESSION_HARD_STOP_HOURS
)

print(
    PIPELINE_NAME,
    {
        "dataset": "AG-ReID.v2",
        "protocol": AG_PROTOCOL,
        "E2_epochs": E2_EPOCHS,
        "E3_epochs": E3_EPOCHS,
        "init": "MSMT17 TransReID -> trainable UAD ViT",
        "feature_dim": TRANSREID_TARGET_EMBED_DIM,
        "target_stride": TRANSREID_TARGET_STRIDE,
        "TIR_distillation": False,
    },
)

PVUAD_AGREIDV2_E2_E3_TRANSREID {'dataset': 'AG-ReID.v2', 'protocol': 'aerial_to_ground', 'E2_epochs': 60, 'E3_epochs': 25, 'init': 'MSMT17 TransReID -> trainable UAD ViT', 'feature_dim': 768, 'target_stride': [16, 16], 'TIR_distillation': False}


## 1. Kiểm tra môi trường

In [2]:
import base64
import csv
import hashlib
import importlib.util
import io
import json
import os
import re
import shlex
import shutil
import subprocess
import sys
import tarfile
import urllib.request
import zipfile
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import torch

if not torch.cuda.is_available():
    raise RuntimeError("Enable a Kaggle GPU accelerator and restart the session.")
if torch.cuda.device_count() < NUM_GPUS:
    raise RuntimeError(
        f"This run expects {NUM_GPUS} GPUs, but Kaggle exposes {torch.cuda.device_count()}. "
        "Choose GPU T4 x2 in Notebook options."
    )

for package, pip_name in [("yacs", "yacs==0.1.8"), ("timm", "timm>=0.9,<2"), ("gdown", "gdown>=5,<6")]:
    if importlib.util.find_spec(package) is None:
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", pip_name],
            check=True,
        )

gpu_rows = []
for index in range(NUM_GPUS):
    props = torch.cuda.get_device_properties(index)
    gpu_rows.append({
        "index": index,
        "name": props.name,
        "memory_GiB": round(props.total_memory / 2**30, 2),
    })
display(pd.DataFrame(gpu_rows))
print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA runtime:", torch.version.cuda)


,index,name,memory_GiB
0,0,Tesla T4,14.56
1,1,Tesla T4,14.56


Python: 3.12.13
PyTorch: 2.10.0+cu128
CUDA runtime: 12.8


## 2. Khóa AG-ReID.v2 và protocol Aerial→Ground — nested-folder fix

AG-ReID.v2 **không để ảnh trực tiếp trong `train_all/query/gallery`**.
Bản official thường có một tầng thư mục identity ở giữa, ví dụ:

`train_all/P0001T04041A0/*.jpg`  
`query/P0000T02140A0/*.jpg`  
`gallery/P0000T02140A0/*.jpg`

Bản v2 này vì vậy:

- tìm Kaggle mount ở cả path user đưa **và** `/kaggle/input/ag-reid-v2`;
- tự tìm dataset root bên dưới `/kaggle/input` nếu mount name khác;
- quét ảnh **recursive** trong `train_all/query/gallery`;
- vẫn hỗ trợ challenge layout `bounding_box_train/exp3_aerial_to_ground/...`;
- bắt buộc audit đúng **807 train IDs**, **808 query IDs**, **808 gallery IDs**;
- filter protocol đúng **C0 aerial query → C2/C3 ground gallery**.

In [3]:
# ============================================================
# 2. Resolve and audit AG-ReID.v2 aerial->ground
#    FIX: official AG-ReID.v2 stores images inside per-ID folders.
# ============================================================

IMAGE_SPLITS = {
    "train", "train_all", "bounding_box_train",
    "query", "gallery", "bounding_box_test",
    "exp3_aerial_to_ground", "exp3_a2g",
    "gallery_add_on",
}

AG_PID_RE = re.compile(r"P(\d+)T(\d+)A(\d+)", re.IGNORECASE)
AG_CAM_RE = re.compile(r"C(\d+)F(\d+)", re.IGNORECASE)
AG_CAM_REMAP = {2: 0, 3: 1, 0: 2}
IMAGE_EXTS = {".jpg", ".jpeg", ".png"}

def image_files(path):
    """Return images recursively; AG-ReID.v2 uses per-identity subfolders."""
    path = Path(path)
    if not path.is_dir():
        return []
    values = []
    for root, _dirs, files in os.walk(path):
        root = Path(root)
        for name in files:
            p = root / name
            if p.suffix.lower() in IMAGE_EXTS:
                values.append(p)
    return sorted(values)

def has_images(path):
    path = Path(path)
    if not path.is_dir():
        return False
    # Cheap recursive early-exit.
    for root, _dirs, files in os.walk(path):
        if any(Path(name).suffix.lower() in IMAGE_EXTS for name in files):
            return True
    return False

def first_image_dir(root, relatives):
    for relative in relatives:
        candidate = Path(root) / relative
        if has_images(candidate):
            return candidate
    return None

def resolve_ag_layout(root):
    root = Path(root)

    train_dir = first_image_dir(root, [
        "train_all",
        "bounding_box_train",
        "train",
    ])

    query_dir = first_image_dir(root, [
        "query",
        "exp3_aerial_to_ground/query",
        "exp3_A2G/query",
    ])

    primary_gallery = first_image_dir(root, [
        "gallery",
        "exp3_aerial_to_ground/bounding_box_test",
        "exp3_aerial_to_ground/gallery",
        "exp3_A2G/bounding_box_test",
        "exp3_A2G/gallery",
        "bounding_box_test",
    ])

    if train_dir is None or query_dir is None or primary_gallery is None:
        return None

    gallery_dirs = [primary_gallery]
    for relative in (
        "gallery_add_on/exp3_aerial_to_ground",
        "gallery_add_on/exp3_A2G",
    ):
        candidate = root / relative
        if has_images(candidate):
            gallery_dirs.append(candidate)

    return train_dir, query_dir, gallery_dirs

def discover_candidate_roots():
    requested = Path(AGREIDV2_INPUT)

    # Kaggle normally mounts a dataset as /kaggle/input/<slug>.
    starting_points = [
        requested,
        Path("/kaggle/input/ag-reid-v2"),
        Path("/kaggle/input/datasets/thienbao1604/ag-reid-v2"),
    ]

    candidates = []

    def add_candidate(path):
        path = Path(path)
        if path.is_dir() and path not in candidates:
            candidates.append(path)

    for start in starting_points:
        add_candidate(start)

    # Search /kaggle/input by directory NAMES only. Do not descend into the
    # large split trees; this keeps discovery fast even with 100k images.
    kaggle_input = Path("/kaggle/input")
    if kaggle_input.is_dir():
        for root, dirs, _files in os.walk(kaggle_input):
            root = Path(root)
            try:
                depth = len(root.relative_to(kaggle_input).parts)
            except ValueError:
                depth = 99

            lower_dirs = {d.lower(): d for d in dirs}

            if any(name in lower_dirs for name in (
                "train_all", "bounding_box_train", "train"
            )):
                add_candidate(root)

            # If this directory itself is a likely AG root, keep it.
            if root.name.lower() in {
                "ag-reid.v2", "ag_reid_v2", "ag-reid-v2", "agreidv2"
            }:
                add_candidate(root)

            # Never recurse into huge image split folders during root discovery.
            dirs[:] = [
                d for d in dirs
                if d.lower() not in {
                    "train_all", "bounding_box_train", "train",
                    "query", "gallery", "bounding_box_test",
                }
            ]

            if depth >= 7:
                dirs[:] = []

    return candidates

candidate_roots = discover_candidate_roots()

resolved = []
for root in candidate_roots:
    layout = resolve_ag_layout(root)
    if layout is not None:
        resolved.append((root, layout))

if not resolved:
    preview = []
    for base in (
        Path("/kaggle/input/ag-reid-v2"),
        Path("/kaggle/input"),
    ):
        if not base.is_dir():
            continue
        for root, dirs, files in os.walk(base):
            root = Path(root)
            try:
                depth = len(root.relative_to(base).parts)
            except ValueError:
                depth = 99
            if depth <= 4:
                preview.append({
                    "path": str(root),
                    "dirs": sorted(dirs)[:20],
                    "image_files_here": sum(
                        Path(name).suffix.lower() in IMAGE_EXTS for name in files
                    ),
                })
            dirs[:] = [
                d for d in dirs
                if d.lower() not in {
                    "train_all", "bounding_box_train", "train",
                    "query", "gallery", "bounding_box_test",
                }
            ]
            if depth >= 4:
                dirs[:] = []
        if preview:
            break

    if preview:
        display(pd.DataFrame(preview).head(100))

    raise FileNotFoundError(
        "Cannot resolve AG-ReID.v2. The notebook checked both the user path "
        f"({AGREIDV2_INPUT}) and normal Kaggle mounts under /kaggle/input. "
        "Expected a root containing train_all/query/gallery (nested identity "
        "folders are supported) or the challenge exp3 layout."
    )

requested = Path(AGREIDV2_INPUT)
resolved.sort(
    key=lambda item: (
        int(item[0] == requested),
        int("ag-reid" in str(item[0]).lower() or "agreid" in str(item[0]).lower()),
        len(item[0].parts),
        str(item[0]),
    ),
    reverse=True,
)
DATA_ROOT, (TRAIN_DIR, QUERY_DIR, GALLERY_DIRS) = resolved[0]

def parse_ag_filename(path):
    name = Path(path).name
    pid_match = AG_PID_RE.search(name)
    cam_match = AG_CAM_RE.search(name)
    if pid_match is None or cam_match is None:
        raise ValueError(f"Unexpected AG-ReID.v2 filename: {name}")

    p, t, a = pid_match.groups()

    # Keep the same identity key as the official AG-ReID tooling/pipeline:
    # P + T + A together identify the instance identity folder.
    pid = int(p + t + a)

    original_cam = int(cam_match.group(1))
    if original_cam not in AG_CAM_REMAP:
        raise ValueError(f"Unexpected camera C{original_cam}: {name}")

    return pid, original_cam, AG_CAM_REMAP[original_cam]

def summarize(paths, allowed_original_cams=None):
    rows = []
    for path in paths:
        pid, original_cam, remapped_cam = parse_ag_filename(path)
        if (
            allowed_original_cams is not None
            and original_cam not in allowed_original_cams
        ):
            continue
        rows.append((path, pid, original_cam, remapped_cam))

    return pd.DataFrame(
        rows,
        columns=["path", "pid", "camera_original", "camera_remapped"],
    )

print("Scanning nested AG-ReID.v2 split folders...")
train_paths = image_files(TRAIN_DIR)
query_paths = image_files(QUERY_DIR)

gallery_paths = []
for directory in GALLERY_DIRS:
    gallery_paths.extend(image_files(directory))
gallery_paths = sorted(set(gallery_paths))

train_frame = summarize(train_paths)
query_frame = summarize(query_paths, allowed_original_cams={0})
gallery_frame = summarize(gallery_paths, allowed_original_cams={2, 3})

if train_frame.empty or query_frame.empty or gallery_frame.empty:
    raise RuntimeError(
        "Resolved AG-ReID.v2 layout contains an empty required split after "
        "Aerial->Ground camera filtering."
    )

audit = pd.DataFrame([
    {
        "split": "train",
        "images": len(train_frame),
        "ids": train_frame["pid"].nunique(),
        "cameras": sorted(train_frame["camera_original"].unique().tolist()),
    },
    {
        "split": "query_aerial_C0",
        "images": len(query_frame),
        "ids": query_frame["pid"].nunique(),
        "cameras": sorted(query_frame["camera_original"].unique().tolist()),
    },
    {
        "split": "gallery_ground_C2_C3",
        "images": len(gallery_frame),
        "ids": gallery_frame["pid"].nunique(),
        "cameras": sorted(gallery_frame["camera_original"].unique().tolist()),
    },
])
display(audit)

if train_frame["pid"].nunique() != 807:
    raise RuntimeError(
        f"Expected 807 train IDs, got {train_frame['pid'].nunique()}. "
        f"Resolved train dir: {TRAIN_DIR}"
    )
if query_frame["pid"].nunique() != 808:
    raise RuntimeError(
        f"Expected 808 aerial query IDs, got {query_frame['pid'].nunique()}. "
        f"Resolved query dir: {QUERY_DIR}"
    )
if gallery_frame["pid"].nunique() != 808:
    raise RuntimeError(
        f"Expected 808 ground gallery IDs, got {gallery_frame['pid'].nunique()}. "
        f"Resolved gallery dirs: {GALLERY_DIRS}"
    )

missing_gallery_ids = sorted(
    set(query_frame["pid"]) - set(gallery_frame["pid"])
)
if missing_gallery_ids:
    raise RuntimeError(
        f"Query IDs missing from ground gallery: {missing_gallery_ids[:20]}"
    )

print("Resolved AG-ReID.v2:", DATA_ROOT)
print("Train dir          :", TRAIN_DIR)
print("Query dir (C0)     :", QUERY_DIR)
print("Gallery dirs (C2/3):", GALLERY_DIRS)

# Link the real dataset root into the name expected by the patched UAD loader.
DATA_LINK_PARENT = Path("/kaggle/working/pvuad_agreid_data")
DATA_LINK_PARENT.mkdir(parents=True, exist_ok=True)
DATA_LINK = DATA_LINK_PARENT / "AG-ReID.v2"

if DATA_LINK.is_symlink():
    DATA_LINK.unlink()
elif DATA_LINK.exists():
    resolved_link = DATA_LINK.resolve()
    if not str(resolved_link).startswith("/kaggle/working/"):
        raise RuntimeError(f"Refusing to remove unsafe path: {resolved_link}")
    shutil.rmtree(DATA_LINK)

DATA_LINK.symlink_to(DATA_ROOT, target_is_directory=True)

AG_TRAIN_ID_COUNT = int(train_frame["pid"].nunique())
AG_QUERY_ID_COUNT = int(query_frame["pid"].nunique())
AG_GALLERY_ID_COUNT = int(gallery_frame["pid"].nunique())

DATASET_AUDIT = {
    "dataset": "AG-ReID.v2",
    "protocol": "aerial_to_ground",
    "root": str(DATA_ROOT),
    "train_dir": str(TRAIN_DIR),
    "query_dir": str(QUERY_DIR),
    "gallery_dirs": [str(p) for p in GALLERY_DIRS],
    "train_images": int(len(train_frame)),
    "train_ids": AG_TRAIN_ID_COUNT,
    "query_images": int(len(query_frame)),
    "query_ids": AG_QUERY_ID_COUNT,
    "gallery_images": int(len(gallery_frame)),
    "gallery_ids": AG_GALLERY_ID_COUNT,
    "camera_remap": {"C2_ground": 0, "C3_ground": 1, "C0_aerial": 2},
    "recursive_split_scan": True,
}
(Path("/kaggle/working") / "agreid_v2_dataset_audit.json").write_text(
    json.dumps(DATASET_AUDIT, indent=2),
    encoding="utf-8",
)
print("UAD data link:", DATA_LINK)

Scanning nested AG-ReID.v2 split folders...


,split,images,ids,cameras
0,train,51530,807,"[0, 2, 3]"
1,query_aerial_C0,4348,808,[0]
2,gallery_ground_C2_C3,19259,808,"[2, 3]"


Resolved AG-ReID.v2: /kaggle/input/datasets/thienbao1604/ag-reid-v2/AG-ReID.v2/AG-ReID.v2
Train dir          : /kaggle/input/datasets/thienbao1604/ag-reid-v2/AG-ReID.v2/AG-ReID.v2/train_all
Query dir (C0)     : /kaggle/input/datasets/thienbao1604/ag-reid-v2/AG-ReID.v2/AG-ReID.v2/query
Gallery dirs (C2/3): [PosixPath('/kaggle/input/datasets/thienbao1604/ag-reid-v2/AG-ReID.v2/AG-ReID.v2/gallery')]
UAD data link: /kaggle/working/pvuad_agreid_data/AG-ReID.v2


## 3. Pin source UAD + patch AG-ReID.v2 + TransReID initialization

Notebook vẫn pin source UAD tại commit gốc của pipeline 11.877, sau đó áp overlay
được đóng gói ngay trong notebook. Overlay thêm:

- loader AG-ReID.v2;
- view-balanced PKM sampler cho RGB aerial/ground;
- dynamic scenario classes = `2 × số modalities` (AG-ReID.v2 = 2);
- source UAD/TransReID 768-D như pipeline cũ;
- multi-session processor và evaluator.

Checkpoint TransReID vẫn là official MSMT17 `vit_transreid_msmt.pth`.

In [4]:
PINNED_UAD_COMMIT = "3e2a07314119402586c76096e87d40420894ad30"
OFFICIAL_REPO_URL = "https://github.com/msm8976/WHU-MARS.git"
WORK_REPO = Path("/kaggle/working/WHU_MARS_official_pinned")
UAD_DIR = WORK_REPO / "CVPR26_UAD"

if WORK_REPO.exists():
    resolved = WORK_REPO.resolve()
    if not str(resolved).startswith("/kaggle/working/"):
        raise RuntimeError(f"Unsafe generated-source path: {resolved}")
    shutil.rmtree(WORK_REPO)

def attached_source_candidates(base="/kaggle/input"):
    candidates = []
    base = Path(base)
    if not base.exists():
        return candidates
    for root, dirs, _files in os.walk(base):
        root_path = Path(root)
        if root_path.name.lower() in IMAGE_SPLITS:
            dirs[:] = []
            continue
        if root_path.name != "CVPR26_UAD" or not (root_path / "train.py").is_file():
            continue
        repo_root = root_path.parent
        declared_commit = None
        manifest_path = root_path / "PVUAD_patch_manifest.json"
        if manifest_path.is_file():
            try:
                declared_commit = json.loads(
                    manifest_path.read_text(encoding="utf-8")
                ).get("base_commit")
            except Exception:
                declared_commit = None
        if (repo_root / ".git").is_dir():
            try:
                declared_commit = subprocess.check_output(
                    ["git", "-C", str(repo_root), "rev-parse", "HEAD"],
                    text=True,
                ).strip()
            except Exception:
                pass
        if declared_commit != PINNED_UAD_COMMIT:
            # Never auto-select an unverified source tree. With no pinned
            # prior output, fall back to cloning the exact official commit.
            dirs[:] = []
            continue
        score = (
            2,
            1 if manifest_path.is_file() else 0,
            1 if "notebooks" in str(repo_root).lower() else 0,
        )
        candidates.append((score, repo_root, declared_commit))
        dirs[:] = []
    return sorted(candidates, key=lambda item: (item[0], str(item[1])), reverse=True)

source = Path(SOURCE_DIR_OVERRIDE) if SOURCE_DIR_OVERRIDE else None
if source is None:
    attached_sources = attached_source_candidates()
    if attached_sources:
        print("Attached UAD source candidates:")
        display(pd.DataFrame([
            {
                "path": str(path),
                "base_commit": commit,
                "selected": index == 0,
            }
            for index, (_score, path, commit) in enumerate(attached_sources)
        ]))
        source = attached_sources[0][1]

if source is not None:
    if (source / "CVPR26_UAD" / "train.py").is_file():
        shutil.copytree(source, WORK_REPO)
    elif source.name == "CVPR26_UAD" and (source / "train.py").is_file():
        WORK_REPO.mkdir(parents=True)
        shutil.copytree(source, UAD_DIR)
    else:
        raise FileNotFoundError(
            f"Attached/override source is not a WHU-MARS source tree: {source}"
        )
    print("Reusing attached UAD source:", source)
else:
    clone_command = [
        "git", "clone", "--no-checkout", "--filter=blob:none",
        OFFICIAL_REPO_URL, str(WORK_REPO),
    ]
    try:
        subprocess.run(clone_command, check=True)
        subprocess.run(
            ["git", "-C", str(WORK_REPO), "checkout", "--detach", PINNED_UAD_COMMIT],
            check=True,
        )
    except subprocess.CalledProcessError as error:
        raise RuntimeError(
            "Cannot clone the pinned official UAD source. Enable Kaggle Internet or attach "
            "the previous notebook output/source and set SOURCE_DIR_OVERRIDE."
        ) from error

if not (UAD_DIR / "train.py").is_file():
    raise FileNotFoundError(f"Missing UAD train.py under {UAD_DIR}")
if (WORK_REPO / ".git").is_dir():
    head = subprocess.check_output(
        ["git", "-C", str(WORK_REPO), "rev-parse", "HEAD"], text=True
    ).strip()
    if head != PINNED_UAD_COMMIT:
        raise RuntimeError(f"Source commit mismatch: expected {PINNED_UAD_COMMIT}, got {head}")

OVERLAY_ARCHIVE_B64 = '...'
overlay_bytes = base64.b64decode(OVERLAY_ARCHIVE_B64)
with tarfile.open(fileobj=io.BytesIO(overlay_bytes), mode="r:gz") as archive:
    members = archive.getmembers()
    for member in members:
        destination = (UAD_DIR / member.name).resolve()
        if not str(destination).startswith(str(UAD_DIR.resolve()) + os.sep):
            raise RuntimeError(f"Unsafe overlay member: {member.name}")
    archive.extractall(UAD_DIR)

OVERLAY_HASHES = {'config/defaults.py': '839faddc24b14d21f073c0a2b588406cd24ac50651304c7d7dcdbe8d7416b89e', 'configs/PVUAD.yml': '2ce61c2c87eced2377c6b290fa5cfc14dc17511eb85dbeee4c18e6054c727440', 'datasets/agreid_v2.py': 'f2dc8f1bb9f85ae5ea7b6d2602c38ddf22bec14b935853773e337b588e27c051', 'datasets/bases.py': '7712bac25aee214f38a913cf3aad1ce4ef33b9a713d7e177ad67ec17932c016d', 'datasets/make_dataloader.py': 'edd99ebc3248edac65a689f79b390cecec14a58e947a1c19fdc44506c159df09', 'datasets/pvuad_sampler.py': '93d21613c90372d10fca9d4b1a0ffbb34ec1a6f4849c9f6eb10671d736261d75', 'datasets/pvuad_transforms.py': '056fc9f25aba6361d4612bcc313f0cc2d175e2b695757f6912e09fc7b5fc61c0', 'datasets/sampler_ddp.py': 'e4c455f7c7afbfac09a503a924c7b09586dbdb8eda6c88735ad1a3e37599bf7c', 'datasets/whu_mars.py': '11d3a0c501912a9887d78601df01afb6495d2f859835a6b69d2079135dba24b3', 'model/backbones/vit_pytorch.py': '723305c1a05e204b744ddb0ff7873b5e02840329e01787200173c9064ed3ad05', 'model/make_model.py': 'bcdfef10d3a75cbb25da27eb688bb6834e97baec8b4221bbef15cf9bf472e3b5', 'processor/processor.py': 'e9a5dc613a374fde37874a4bf838789c7b359ff44c2e0d051db4e7e8fb987d7c', 'train.py': '8da1552749d67272ca6d8078ac3dd8c1df4183924d7795a6e7b0355eecf18ff5'}
for relative, expected_hash in OVERLAY_HASHES.items():
    actual_hash = hashlib.sha256((UAD_DIR / relative).read_bytes()).hexdigest()
    if actual_hash != expected_hash:
        raise RuntimeError(f"Overlay hash mismatch: {relative}")
patch_manifest = {
    "base_repo": OFFICIAL_REPO_URL,
    "base_commit": PINNED_UAD_COMMIT,
    "overlay_variant": "agreidv2_a2g_transreid_init_trainable_768_global_b1_v6_ddpevalfix",
    "overlay_sha256": OVERLAY_HASHES,
}
(UAD_DIR / "PVUAD_patch_manifest.json").write_text(
    json.dumps(patch_manifest, indent=2), encoding="utf-8"
)
subprocess.run([sys.executable, "-m", "compileall", "-q", str(UAD_DIR)], check=True)
print("Pinned and patched source:", UAD_DIR)



TRANSREID_MODEL_NAME = "vit_transreid_msmt.pth"
TRANSREID_GDRIVE_ID = "1x6Na97ycxS0t2Dn_0iRKWe1U5ccIqASK"
TRANSREID_SOURCE_STRIDE = [12, 12]

def find_named_file_without_images(base, filename):
    base = Path(base)
    if not base.exists():
        return []
    found = []
    for root, dirs, files in os.walk(base):
        root_path = Path(root)
        if filename in files:
            found.append(root_path / filename)
        if root_path.name.lower() in IMAGE_SPLITS:
            dirs[:] = []
    return sorted(found, key=lambda item: str(item))


def torch_load_cpu(path):
    try:
        return torch.load(path, map_location="cpu", weights_only=False)
    except TypeError:
        return torch.load(path, map_location="cpu")


def unwrap_checkpoint_state(payload):
    if isinstance(payload, dict):
        if "model" in payload and isinstance(payload["model"], dict):
            payload = payload["model"]
        elif "state_dict" in payload and isinstance(payload["state_dict"], dict):
            payload = payload["state_dict"]
    if not isinstance(payload, dict):
        raise ValueError("Checkpoint does not contain a state_dict")
    return {str(key).replace("module.", ""): value for key, value in payload.items()}


TRANSREID_CACHE = Path("/kaggle/working") / TRANSREID_MODEL_NAME
checkpoint_override = TRANSREID_CHECKPOINT_OVERRIDE or PRETRAIN_PATH_OVERRIDE
if checkpoint_override:
    source_checkpoint = Path(checkpoint_override)
    if not source_checkpoint.is_file():
        raise FileNotFoundError(source_checkpoint)
    if source_checkpoint.resolve() != TRANSREID_CACHE.resolve():
        shutil.copy2(source_checkpoint, TRANSREID_CACHE)
elif TRANSREID_CACHE.is_file():
    pass
else:
    attached = find_named_file_without_images("/kaggle/input", TRANSREID_MODEL_NAME)
    if attached:
        print("Caching attached pretrained TransReID checkpoint...")
        shutil.copy2(attached[0], TRANSREID_CACHE)
    else:
        print("Downloading official MSMT17 TransReID* ViT checkpoint with gdown...")
        try:
            import gdown
            result = gdown.download(
                id=TRANSREID_GDRIVE_ID,
                output=str(TRANSREID_CACHE),
                quiet=False,
            )
            if not result:
                raise RuntimeError("gdown returned no output path")
        except Exception as error:
            raise RuntimeError(
                "Cannot obtain vit_transreid_msmt.pth. Enable Kaggle Internet, "
                "or attach the official checkpoint as a Kaggle Dataset and set "
                "TRANSREID_CHECKPOINT_OVERRIDE."
            ) from error

if not TRANSREID_CACHE.is_file():
    raise FileNotFoundError(TRANSREID_CACHE)
if TRANSREID_CACHE.stat().st_size < 300_000_000:
    raise RuntimeError(
        "TransReID checkpoint is unexpectedly small; this is likely an HTML/"
        "partial download: {} bytes".format(TRANSREID_CACHE.stat().st_size)
    )

payload = torch_load_cpu(TRANSREID_CACHE)
tr_state = unwrap_checkpoint_state(payload)
required_transreid_keys = {
    "base.cls_token",
    "base.pos_embed",
    "base.patch_embed.proj.weight",
    "base.blocks.0.attn.qkv.weight",
    "base.blocks.11.attn.qkv.weight",
    "base.norm.weight",
    # These two are validated only to make sure this is the official full
    # TransReID checkpoint; they are intentionally NOT transplanted into UAD.
    "base.sie_embed",
    "b1.0.attn.qkv.weight",
    "b2.0.attn.qkv.weight",
}
missing = sorted(required_transreid_keys - set(tr_state))
if missing:
    raise RuntimeError(
        "Attached/downloaded file is not the expected full MSMT17 TransReID "
        f"checkpoint; missing keys: {missing}"
    )
if tr_state["base.cls_token"].shape[-1] != 768:
    raise RuntimeError("Expected ViT-B hidden dimension 768.")
if tuple(tr_state["base.sie_embed"].shape) != (15, 1, 768):
    raise RuntimeError(
        "Unexpected MSMT17 source SIE shape: "
        + str(tuple(tr_state["base.sie_embed"].shape))
    )
print(
    "Validated pretrained TransReID source:",
    {
        "path": str(TRANSREID_CACHE),
        "size_MiB": round(TRANSREID_CACHE.stat().st_size / 2**20, 1),
        "source_pos_embed": tuple(tr_state["base.pos_embed"].shape),
        "source_SIE": tuple(tr_state["base.sie_embed"].shape),
        "source_has_JPM": True,
        "target_embedding_dim": 768,
        "target_stride": TRANSREID_TARGET_STRIDE,
        "target_SIE": False,
        "target_JPM": False,
        "backbone_will_be_trainable": True,
    },
)
del payload, tr_state

TRANSREID_PATH = TRANSREID_CACHE
PRETRAIN_PATH = TRANSREID_PATH
print("Persistent TransReID initialization checkpoint:", PRETRAIN_PATH)


if RUN_TRANSREID_INIT_SMOKE_TEST:
    print("Running AG-ReID.v2 trainable 768-D TransReID-initialized UAD smoke test...")
    TRANSREID_INIT_AUDIT_PATH = Path(
        "/kaggle/working/transreid_trainable_init_audit.json"
    )
    smoke_script = r"""
import json
import sys
from pathlib import Path
import torch

uad_dir = Path(sys.argv[1])
checkpoint = Path(sys.argv[2])
audit_path = Path(sys.argv[3])
sys.path.insert(0, str(uad_dir))

from config import cfg
from model import make_model

def load_state(path):
    try:
        payload = torch.load(path, map_location="cpu", weights_only=False)
    except TypeError:
        payload = torch.load(path, map_location="cpu")
    if isinstance(payload, dict):
        if "model" in payload and isinstance(payload["model"], dict):
            payload = payload["model"]
        elif "state_dict" in payload and isinstance(payload["state_dict"], dict):
            payload = payload["state_dict"]
    return {str(k).replace("module.", ""): v for k, v in payload.items()}

cfg.merge_from_file(str(uad_dir / "configs/PVUAD.yml"))
cfg.MODEL.PRETRAIN_CHOICE = "transreid"
cfg.MODEL.PRETRAIN_PATH = str(checkpoint)
cfg.MODEL.STRIDE_SIZE = [16, 16]
cfg.MODEL.SIE_CAMERA = False
cfg.MODEL.SIE_VIEW = False
cfg.DATASETS.NAMES = "AG-ReID.v2"
cfg.DATASETS.MODALITIES = ["RGB"]
cfg.PVUAD.GROUND_MAX_CAMID = 1
cfg.PVUAD.VFPROCA = True
cfg.PVUAD.LOCAL_CONSISTENCY = False
cfg.PVUAD.TIR_DISTILLATION = False

model = make_model(cfg, num_class=807, camera_num=3, view_num=0).cuda().train()
if model.transreid_init_audit is None:
    raise RuntimeError("Missing TransReID initialization audit")
if model.in_planes != 768:
    raise RuntimeError("UAD backbone must remain 768-D")
if hasattr(model.base, "sie_embed"):
    raise RuntimeError("Source-camera SIE must not be copied into target UAD")
if not all(parameter.requires_grad for parameter in model.base.parameters()):
    raise RuntimeError("TransReID-initialized backbone is unexpectedly frozen")

source = load_state(checkpoint)
source_qkv = source["base.blocks.0.attn.qkv.weight"]
target_qkv = model.base.blocks[0].attn.qkv.weight.detach().cpu()
init_qkv_max_abs = float((source_qkv - target_qkv).abs().max())
source_global_final_qkv = source["b1.0.attn.qkv.weight"]
target_final_qkv = model.base.blocks[11].attn.qkv.weight.detach().cpu()
global_final_qkv_max_abs = float(
    (source_global_final_qkv - target_final_qkv).abs().max()
)
if init_qkv_max_abs > 1e-7 or global_final_qkv_max_abs > 1e-7:
    raise RuntimeError(
        "TransReID weights were not transplanted exactly: "
        f"shared_qkv={init_qkv_max_abs}, global_final_qkv={global_final_qkv_max_abs}"
    )

SMOKE_BATCH = 2
images = [
    torch.randn(SMOKE_BATCH, 3, 256, 128, device="cuda"),
]
output = model(x=images, mode=0)
cls_score, global_feat = output[0], output[1]
if tuple(global_feat.shape) != (SMOKE_BATCH, 768):
    raise RuntimeError(f"Unexpected UAD feature shape: {tuple(global_feat.shape)}")
loss = cls_score.float().square().mean() + 0.01 * global_feat[:, :16].float().square().mean()
loss.backward()

qkv_grad = model.base.blocks[0].attn.qkv.weight.grad
patch_grad = model.base.patch_embed.proj.weight.grad
if qkv_grad is None or patch_grad is None:
    raise RuntimeError("No gradient reached the TransReID-initialized backbone")
qkv_grad_norm = float(qkv_grad.float().norm().detach().cpu())
patch_grad_norm = float(patch_grad.float().norm().detach().cpu())
if not (qkv_grad_norm > 0 and patch_grad_norm > 0):
    raise RuntimeError(
        f"Backbone gradients must be nonzero; qkv={qkv_grad_norm}, "
        f"patch={patch_grad_norm}"
    )

audit = dict(model.transreid_init_audit)
audit.update({
    "smoke_test": "OK",
    "feature_shape": list(global_feat.shape),
    "smoke_batch_size": SMOKE_BATCH,
    "batchnorm_train_mode_checked": True,
    "init_qkv_max_abs": init_qkv_max_abs,
    "global_final_qkv_max_abs": global_final_qkv_max_abs,
    "qkv_grad_norm": qkv_grad_norm,
    "patch_grad_norm": patch_grad_norm,
    "backbone_all_trainable": True,
    "source_sie_used": False,
    "source_jpm_local_branches_used": False,
    "source_jpm_global_final_branch_used": True,
    "target_stride": [16, 16],
})
audit_path.write_text(json.dumps(audit, indent=2), encoding="utf-8")
print("TRANSREID_TRAINABLE_INIT_SMOKE_OK", audit)
"""
    smoke_env = os.environ.copy()
    smoke_env["CUDA_VISIBLE_DEVICES"] = "0"
    smoke_env["PYTHONPATH"] = str(UAD_DIR) + os.pathsep + smoke_env.get(
        "PYTHONPATH", ""
    )
    result = subprocess.run(
        [
            sys.executable,
            "-c",
            smoke_script,
            str(UAD_DIR),
            str(TRANSREID_PATH),
            str(TRANSREID_INIT_AUDIT_PATH),
        ],
        cwd=UAD_DIR,
        env=smoke_env,
        text=True,
        capture_output=True,
    )
    print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)
        raise RuntimeError(
            "Trainable TransReID initialization smoke test failed. "
            "Do not start E2 until the initialization path is fixed."
        )


Attached UAD source candidates:


fatal: detected dubious ownership in repository at '/kaggle/input/notebooks/thienbao1604/pvuad-agreidv2-a2g-e2-60-e3-25-transreidfd4dd43eb8/WHU_MARS_official_pinned'
To add an exception for this directory, call:

	git config --global --add safe.directory /kaggle/input/notebooks/thienbao1604/pvuad-agreidv2-a2g-e2-60-e3-25-transreidfd4dd43eb8/WHU_MARS_official_pinned


,path,base_commit,selected
0,/kaggle/input/notebooks/thienbao1604/pvuad-agr...,3e2a07314119402586c76096e87d40420894ad30,True


Reusing attached UAD source: /kaggle/input/notebooks/thienbao1604/pvuad-agreidv2-a2g-e2-60-e3-25-transreidfd4dd43eb8/WHU_MARS_official_pinned
Pinned and patched source: /kaggle/working/WHU_MARS_official_pinned/CVPR26_UAD


/tmp/ipykernel_23/3906047045.py:116: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  archive.extractall(UAD_DIR)


Caching attached pretrained TransReID checkpoint...
Validated pretrained TransReID source: {'path': '/kaggle/working/vit_transreid_msmt.pth', 'size_MiB': 399.8, 'source_pos_embed': (1, 211, 768), 'source_SIE': (15, 1, 768), 'source_has_JPM': True, 'target_embedding_dim': 768, 'target_stride': [16, 16], 'target_SIE': False, 'target_JPM': False, 'backbone_will_be_trainable': True}
Persistent TransReID initialization checkpoint: /kaggle/working/vit_transreid_msmt.pth
Running AG-ReID.v2 trainable 768-D TransReID-initialized UAD smoke test...
using Transformer_type: vit_base_in as a backbone
using stride: [16, 16], and patch number is num_y16 * num_x8
TransReID -> trainable UAD backbone initialization: {'checkpoint': '/kaggle/working/vit_transreid_msmt.pth', 'loaded_tensor_count': 152, 'target_tensor_count': 152, 'mismatch_count': 0, 'resized_pos_embed': {'source_shape': [1, 211, 768], 'target_shape': [1, 129, 768], 'target_grid': [16, 8]}, 'source_sie_loaded': False, 'source_jpm_local_bran

## 4. View audit + multi-session state machine

Notebook audit số identity có cả ground/aerial trong train set rồi tự chọn:
**E3 resume → E2 completed → E2 resume → E2 fresh**.

Pipeline tag mới hoàn toàn tách khỏi output WHU-MARS và reranking cũ, nên sẽ
không resume nhầm checkpoint dataset khác.

In [5]:
FILENAME_RE = AG_PID_RE

PIPELINE_ROOT = Path("/kaggle/working/pvuad_runs") / PIPELINE_TAG
E2_RUN_DIR = PIPELINE_ROOT / "e2_final"
E3_RUN_DIR = PIPELINE_ROOT / "e3_tdh"
PIPELINE_ROOT.mkdir(parents=True, exist_ok=True)
E2_RUN_DIR.mkdir(parents=True, exist_ok=True)
E3_RUN_DIR.mkdir(parents=True, exist_ok=True)
PIPELINE_STATE_PATH = PIPELINE_ROOT / "pipeline_state.json"

INIT_AUDIT_SOURCE = Path("/kaggle/working/transreid_trainable_init_audit.json")
INIT_AUDIT_DEST = PIPELINE_ROOT / "transreid_trainable_init_audit.json"
if INIT_AUDIT_SOURCE.is_file() and not INIT_AUDIT_DEST.is_file():
    shutil.copy2(INIT_AUDIT_SOURCE, INIT_AUDIT_DEST)


def read_json(path):
    path = Path(path)
    if not path.is_file():
        return {}
    try:
        return json.loads(path.read_text(encoding="utf-8"))
    except Exception as error:
        print("Ignoring unreadable JSON:", path, error)
        return {}


def write_pipeline_state(status, active_stage, **extra):
    payload = {
        "pipeline": PIPELINE_NAME,
        "pipeline_tag": PIPELINE_TAG,
        "status": status,
        "active_stage": active_stage,
        "updated_at_unix": _time.time(),
        "notebook_started_unix": NOTEBOOK_SESSION_STARTED_UNIX,
        "session_deadline_unix": SESSION_DEADLINE_UNIX,
        "remaining_minutes": max(
            0.0, (SESSION_DEADLINE_UNIX - _time.time()) / 60.0
        ),
        **extra,
    }
    temporary = PIPELINE_STATE_PATH.with_suffix(".json.tmp")
    temporary.write_text(json.dumps(payload, indent=2), encoding="utf-8")
    os.replace(temporary, PIPELINE_STATE_PATH)
    return payload


def remaining_minutes():
    return max(0.0, (SESSION_DEADLINE_UNIX - _time.time()) / 60.0)


def copy_if_present(source_dir, destination_dir, filenames):
    if source_dir is None:
        return
    source_dir = Path(source_dir)
    destination_dir = Path(destination_dir)
    destination_dir.mkdir(parents=True, exist_ok=True)
    for filename in filenames:
        source_path = source_dir / filename
        destination_path = destination_dir / filename
        if source_path.is_file() and source_path.resolve() != destination_path.resolve():
            if not destination_path.exists():
                print("Preserving:", filename)
                shutil.copy2(source_path, destination_path)


def status_progress(status):
    return status.get("progress") or {}


def candidate_sort_key(candidate):
    status = candidate["status"]
    progress = status_progress(status)
    path_text = str(candidate["root"]).lower()
    return (
        int(PIPELINE_TAG.lower() in path_text),
        int(status.get("status") == "completed"),
        int(progress.get("epoch") or 0),
        int(bool(progress.get("epoch_complete", False))),
        int(progress.get("next_iteration") or 0),
        int(status.get("global_step") or 0),
        float(status.get("saved_at_unix") or candidate["checkpoint"].stat().st_mtime),
    )


def scan_stage_candidates(base, experiment):
    base = Path(base)
    if not base.exists():
        return []
    candidates = []
    for root, dirs, files in os.walk(base):
        root_path = Path(root)
        if root_path.name.lower() in IMAGE_SPLITS:
            dirs[:] = []
            continue
        if "pvuad_last_checkpoint.pth" not in files:
            continue
        checkpoint = root_path / "pvuad_last_checkpoint.pth"
        status = read_json(root_path / "session_status.json")
        summary = read_json(root_path / "training_summary.json")
        manifest = read_json(root_path / "run_manifest.json")
        declared = status.get("experiment", summary.get("experiment"))
        lowered = str(root_path).lower()
        legacy_match = (
            experiment == "E2" and "final_e2" in lowered
        ) or (
            experiment == "E3-TDH" and "e3_tdh" in lowered
        )
        if declared != experiment and not legacy_match:
            continue
        if status.get("stage", summary.get("stage")) not in (None, "final"):
            continue
        # Reject a manifest that clearly belongs to a different batch/LR
        # recipe. Missing legacy fields are tolerated, but contradictory
        # fields are not.
        recipe = manifest.get("recipe") or manifest.get("effective_training") or {}
        expected = (
            {"global_batch": E2_GLOBAL_BATCH, "epochs": E2_EPOCHS}
            if experiment == "E2"
            else {"global_batch": E3_GLOBAL_BATCH, "epochs": E3_EPOCHS}
        )
        incompatible = any(
            key in recipe and int(recipe[key]) != int(value)
            for key, value in expected.items()
        )
        lr_value = recipe.get("base_lr", recipe.get("backbone_lr"))
        expected_lr = E2_BASE_LR if experiment == "E2" else E3_BASE_LR
        if lr_value is not None and abs(float(lr_value) - expected_lr) > 1e-12:
            incompatible = True
        if incompatible:
            print("Skipping incompatible stage candidate:", root_path)
            continue
        candidates.append({
            "root": root_path,
            "checkpoint": checkpoint,
            "model": root_path / "pvuad_best_model.pth",
            "last_model": root_path / "pvuad_last_model.pth",
            "status": status,
            "summary": summary,
            "manifest": manifest,
        })
        dirs[:] = []
    return sorted(candidates, key=candidate_sort_key, reverse=True)


def candidate_table(candidates, label):
    if not candidates:
        return
    print(label)
    display(pd.DataFrame([{
        "root": str(item["root"]),
        "status": item["status"].get("status", "legacy_checkpoint"),
        "epoch": status_progress(item["status"]).get("epoch"),
        "next_iteration": status_progress(item["status"]).get("next_iteration"),
        "global_step": item["status"].get("global_step"),
        "best_mAP": item["status"].get("best_mAP_percent"),
        "selected": index == 0,
    } for index, item in enumerate(candidates)]))



# View-coverage audit is invariant across sessions. Reuse it when possible.
PAIRING_AUDIT_PATH = PIPELINE_ROOT / "pairing_audit.csv"
pairing_frame = None

if RUN_METADATA_PAIRING_CHECK:
    by_pid = defaultdict(lambda: {"ground": 0, "aerial": 0})
    for row in train_frame.itertuples():
        if int(row.camera_original) == 0:
            by_pid[int(row.pid)]["aerial"] += 1
        else:
            by_pid[int(row.pid)]["ground"] += 1

    pairing_rows = []
    for pid in sorted(by_pid):
        ground = int(by_pid[pid]["ground"])
        aerial = int(by_pid[pid]["aerial"])
        pairing_rows.append({
            "pid": int(pid),
            "paired_frames": int(ground + aerial),
            "ground": ground,
            "aerial": aerial,
            "view_coverage": (
                "Ground+Aerial" if ground > 0 and aerial > 0
                else "Ground-only" if ground > 0
                else "Aerial-only" if aerial > 0
                else "No sample"
            ),
        })

    pairing_frame = pd.DataFrame(pairing_rows)
    if len(pairing_frame) != AG_TRAIN_ID_COUNT:
        raise RuntimeError(
            f"Expected {AG_TRAIN_ID_COUNT} train identities in view audit, "
            f"got {len(pairing_frame)}"
        )
    pairing_frame.to_csv(PAIRING_AUDIT_PATH, index=False)
    print("AG-ReID.v2 train view coverage:")
    display(
        pairing_frame["view_coverage"]
        .value_counts(dropna=False)
        .rename_axis("view_coverage")
        .reset_index(name="identities")
    )


if E3_RESUME_PATH_OVERRIDE:
    E3_CANDIDATES = [{
        "root": Path(E3_RESUME_PATH_OVERRIDE).parent,
        "checkpoint": Path(E3_RESUME_PATH_OVERRIDE),
        "model": Path(E3_RESUME_PATH_OVERRIDE).parent / "pvuad_best_model.pth",
        "last_model": Path(E3_RESUME_PATH_OVERRIDE).parent / "pvuad_last_model.pth",
        "status": read_json(Path(E3_RESUME_PATH_OVERRIDE).parent / "session_status.json"),
        "summary": {},
        "manifest": read_json(Path(E3_RESUME_PATH_OVERRIDE).parent / "run_manifest.json"),
    }]
else:
    E3_CANDIDATES = scan_stage_candidates(E3_RUN_DIR, "E3-TDH")
    if AUTO_RESUME:
        external_e3 = scan_stage_candidates("/kaggle/input", "E3-TDH")
        external_e3 = [
            item for item in external_e3
            if PIPELINE_TAG.lower() in str(item["root"]).lower()
        ]
        E3_CANDIDATES += external_e3
        unique = {str(item["checkpoint"]): item for item in E3_CANDIDATES}
        E3_CANDIDATES = sorted(unique.values(), key=candidate_sort_key, reverse=True)

if E2_RESUME_PATH_OVERRIDE:
    E2_CANDIDATES = [{
        "root": Path(E2_RESUME_PATH_OVERRIDE).parent,
        "checkpoint": Path(E2_RESUME_PATH_OVERRIDE),
        "model": Path(E2_RESUME_PATH_OVERRIDE).parent / "pvuad_best_model.pth",
        "last_model": Path(E2_RESUME_PATH_OVERRIDE).parent / "pvuad_last_model.pth",
        "status": read_json(Path(E2_RESUME_PATH_OVERRIDE).parent / "session_status.json"),
        "summary": {},
        "manifest": read_json(Path(E2_RESUME_PATH_OVERRIDE).parent / "run_manifest.json"),
    }]
else:
    E2_CANDIDATES = scan_stage_candidates(E2_RUN_DIR, "E2")
    if AUTO_RESUME:
        external_e2 = scan_stage_candidates("/kaggle/input", "E2")
        if not ALLOW_LEGACY_E2_BOOTSTRAP:
            external_e2 = [
                item for item in external_e2
                if PIPELINE_TAG.lower() in str(item["root"]).lower()
            ]
        E2_CANDIDATES += external_e2
        unique = {str(item["checkpoint"]): item for item in E2_CANDIDATES}
        E2_CANDIDATES = sorted(unique.values(), key=candidate_sort_key, reverse=True)

if E2_COMPLETED_DIR_OVERRIDE:
    override_root = Path(E2_COMPLETED_DIR_OVERRIDE)
    E2_CANDIDATES.insert(0, {
        "root": override_root,
        "checkpoint": override_root / "pvuad_last_checkpoint.pth",
        "model": override_root / "pvuad_best_model.pth",
        "last_model": override_root / "pvuad_last_model.pth",
        "status": read_json(override_root / "session_status.json"),
        "summary": read_json(override_root / "training_summary.json"),
        "manifest": read_json(override_root / "run_manifest.json"),
    })

candidate_table(E3_CANDIDATES, "E3 resume candidates (highest priority):")
candidate_table(E2_CANDIDATES, "E2 candidates:")

SELECTED_E3 = E3_CANDIDATES[0] if E3_CANDIDATES else None
SELECTED_E2 = E2_CANDIDATES[0] if E2_CANDIDATES else None

# Preserve compact E2 evidence even when a completed legacy E2 is used only as
# the warm-start for E3. Large model/checkpoint files stay at their selected
# input path until E3 has created its self-contained teacher/resume artifacts.
if SELECTED_E2 is not None:
    copy_if_present(
        SELECTED_E2["root"], E2_RUN_DIR,
        [
            "metrics_history.csv", "session_status.json", "training_summary.json",
            "pvuad_last_model.pth", "pvuad_best_model.pth",
            "pairing_audit.csv", "ablation_summary.csv", "effective_command.txt",
        ],
    )

if SELECTED_E3 is not None:
    ACTIVE_STAGE = "E3"
elif SELECTED_E2 is not None and SELECTED_E2["status"].get("status") == "completed":
    ACTIVE_STAGE = "E3"
elif SELECTED_E2 is not None:
    ACTIVE_STAGE = "E2"
else:
    ACTIVE_STAGE = "E2"

print("Selected pipeline stage:", ACTIVE_STAGE)
write_pipeline_state(
    "ready", ACTIVE_STAGE,
    selected_e2=str(SELECTED_E2["root"]) if SELECTED_E2 else None,
    selected_e3=str(SELECTED_E3["root"]) if SELECTED_E3 else None,
)

(PIPELINE_ROOT / "RESUME_NEXT_SESSION.txt").write_text(
    "PV-UAD AG-ReID.v2 A2G TransReID-init E2 -> E3 multi-session pipeline\n\n"
    "1. A partial session ends normally with e2_paused or e3_paused.\n"
    "2. Start every run with Save Version -> Save & Run All; Kaggle then commits "
    "the output automatically after this notebook exits cleanly.\n"
    "3. In a new session, add the immediately preceding version output as Input.\n"
    "4. Use this exact notebook, keep Settings unchanged, and Run All.\n"
    "5. The state machine resumes E3 first, otherwise completed/partial E2.\n\n"
    f"Pipeline folder: pvuad_runs/{PIPELINE_TAG}\n",
    encoding="utf-8",
)
print("Pipeline output:", PIPELINE_ROOT)


AG-ReID.v2 train view coverage:


,view_coverage,identities
0,Ground+Aerial,807


E2 candidates:


,root,status,epoch,next_iteration,global_step,best_mAP,selected
0,/kaggle/input/notebooks/thienbao1604/pvuad-agr...,completed,60,0,186202,72.677799,True


Preserving: metrics_history.csv
Preserving: session_status.json
Preserving: training_summary.json
Preserving: pvuad_last_model.pth
Preserving: pvuad_best_model.pth
Preserving: effective_command.txt
Selected pipeline stage: E3
Pipeline output: /kaggle/working/pvuad_runs/agreidv2_a2g_e2_60_e3_25_transreid768_b24_v5_manifestfix


## 5. Train E2=60 → E3=25 → evaluate Aerial→Ground

- **E2 fresh:** TransReID MSMT17 → UAD ViT 768-D → AG-ReID.v2.
- **E2→E3:** dùng **last E2 epoch 60 model + last E2 prototype checkpoint**,
  bảo đảm model/prototype cùng epoch.
- **E3:** TIR loss tắt; giữ feature preservation và scenario-hard GPD phù hợp
  với hai view ground/aerial.
- **Final:** flip-TTA cho cả best E3 và fixed epoch-25 E3 khi cần.

In [6]:
environment = os.environ.copy()
environment.update({
    "CUDA_VISIBLE_DEVICES": ",".join(map(str, range(NUM_GPUS))),
    "PYTHONPATH": str(UAD_DIR) + os.pathsep + environment.get("PYTHONPATH", ""),
    "PYTHONUNBUFFERED": "1",
    "OMP_NUM_THREADS": "2",
    "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True",
    "TORCH_NCCL_ASYNC_ERROR_HANDLING": "1",
})


def stage_status(run_dir):
    return read_json(Path(run_dir) / "session_status.json")


def display_stage_status(name, status):
    progress = status_progress(status)
    display(pd.DataFrame([{
        "stage": name,
        "status": status.get("status"),
        "reason": status.get("reason"),
        "epoch": progress.get("epoch"),
        "next_iteration": progress.get("next_iteration"),
        "epoch_complete": progress.get("epoch_complete"),
        "global_step": status.get("global_step"),
        "best_mAP": status.get("best_mAP_percent"),
        "remaining_minutes": round(remaining_minutes(), 1),
    }]))


def run_command_checked(command, run_dir, stage_name):
    command_text = shlex.join(command)
    (Path(run_dir) / "effective_command.txt").write_text(
        command_text + "\n", encoding="utf-8"
    )
    print("\nLaunching", stage_name)
    print(command_text)
    if not RUN_TRAINING:
        print("RUN_TRAINING=False — command prepared but not launched.")
        return {}
    result = subprocess.run(command, cwd=UAD_DIR, env=environment, check=False)
    if result.returncode != 0:
        # A DDP/NCCL process can abort after a periodic checkpoint has already
        # been written. Treat that as a resumable session rather than throwing
        # away hours of work. A missing/invalid checkpoint still fails hard.
        checkpoint = Path(run_dir) / "pvuad_last_checkpoint.pth"
        status_path = Path(run_dir) / "session_status.json"
        recovered_status = stage_status(run_dir) if status_path.is_file() else {}
        if (
            checkpoint.is_file()
            and recovered_status.get("status")
            in {"running_checkpoint", "paused_time_limit"}
        ):
            print(
                f"{stage_name} subprocess exited with code {result.returncode}, "
                "but a valid resumable checkpoint exists. "
                "Ending this notebook cleanly so the next Kaggle session can resume."
            )
            write_pipeline_state(
                "recoverable_stage_crash", stage_name,
                returncode=result.returncode,
                run_dir=str(run_dir),
                checkpoint=str(checkpoint),
            )
            display_stage_status(stage_name, recovered_status)
            return recovered_status
        write_pipeline_state(
            "failed", stage_name,
            returncode=result.returncode,
            run_dir=str(run_dir),
        )
        raise subprocess.CalledProcessError(result.returncode, command)
    required = [
        Path(run_dir) / "pvuad_last_checkpoint.pth",
        Path(run_dir) / "session_status.json",
    ]
    missing = [str(path) for path in required if not path.is_file()]
    if missing:
        raise RuntimeError(f"{stage_name} ended without required outputs: {missing}")
    status = stage_status(run_dir)
    if status.get("status") not in {
        "paused_time_limit", "completed", "running_checkpoint"
    }:
        raise RuntimeError(f"Unexpected {stage_name} status: {status}")
    display_stage_status(stage_name, status)
    return status


def make_stage_manifest(run_dir, stage_name, recipe, resume_path=None, **extra):
    payload = {
        "pipeline": PIPELINE_NAME,
        "pipeline_tag": PIPELINE_TAG,
        "stage_name": stage_name,
        "recipe": recipe,
        "dataset_variant": "AG-ReID.v2",
        "dataset_protocol": "aerial_to_ground",
        "dataset_root": str(DATA_ROOT),
        "dataset_counts": DATASET_AUDIT,
        "source_commit": PINNED_UAD_COMMIT,
        "overlay_sha256": OVERLAY_HASHES,
        "resume_path": str(resume_path) if resume_path else None,
        "python": sys.version,
        "torch": torch.__version__,
        "cuda": torch.version.cuda,
        "gpus": gpu_rows,
        "seed": RANDOM_SEED,
        "session_guard": {
            "notebook_started_unix": NOTEBOOK_SESSION_STARTED_UNIX,
            "session_deadline_unix": SESSION_DEADLINE_UNIX,
            "hard_stop_hours": SESSION_HARD_STOP_HOURS,
            "stop_reserve_minutes": SESSION_STOP_RESERVE_MINUTES,
            "checkpoint_every_minutes": CHECKPOINT_EVERY_MINUTES,
            "time_check_every_steps": TIME_CHECK_EVERY_STEPS,
            "min_eval_remaining_minutes": MIN_EVAL_REMAINING_MINUTES,
        },
        **extra,
    }
    manifest_path = Path(run_dir) / "run_manifest.json"
    manifest_path.write_text(json.dumps(payload, indent=2), encoding="utf-8")
    shutil.copy2(
        UAD_DIR / "PVUAD_patch_manifest.json",
        Path(run_dir) / "PVUAD_patch_manifest.json",
    )
    return manifest_path


def e2_command(resume_path, strict_resume=True):
    # Fresh E2 receives pretrained TransReID backbone initialization exactly
    # once. Resume sessions reconstruct the model from the saved UAD checkpoint,
    # so they must NOT apply the source initialization again.
    pretrain_choice = "none" if resume_path else "transreid"
    recipe = {
        "experiment": "E2", "stage": "final", "epochs": E2_EPOCHS,
        "global_batch": E2_GLOBAL_BATCH, "base_lr": E2_BASE_LR,
        "view_balanced_sampler": True, "vfproca": True,
        "local_consistency": False, "hard_gpd": False,
        "dataset": "AG-ReID.v2", "protocol": "aerial_to_ground",
        "initialization": "MSMT17_TransReID_trainable_backbone",
        "pretrain_choice_this_session": pretrain_choice,
        "embedding_dim": 768,
        "backbone_stride": [16, 16],
        "source_sie_used": False,
        "source_jpm_local_branches_used": False,
        "source_jpm_global_final_branch_used": True,
    }
    manifest_path = make_stage_manifest(
        E2_RUN_DIR, "E2", recipe, resume_path=resume_path
    )
    opts = [
        "MODEL.DIST_TRAIN", "True",
        "MODEL.PRETRAIN_CHOICE", pretrain_choice,
        "MODEL.PRETRAIN_PATH", str(PRETRAIN_PATH),
        "MODEL.STRIDE_SIZE", "[16, 16]",
        "MODEL.SIE_CAMERA", "False",
        "MODEL.SIE_VIEW", "False",
        "DATASETS.NAMES", "AG-ReID.v2",
        "DATASETS.MODALITIES", "['RGB']",
        "DATASETS.ROOT_DIR", str(DATA_LINK_PARENT),
        "DATALOADER.NUM_WORKERS", str(NUM_WORKERS),
        "SOLVER.SEED", str(RANDOM_SEED),
        "SOLVER.MAX_EPOCHS", str(E2_EPOCHS),
        "SOLVER.IMS_PER_BATCH", str(E2_GLOBAL_BATCH),
        "SOLVER.BASE_LR", str(E2_BASE_LR),
        "SOLVER.EVAL_PERIOD", "10",
        "SOLVER.CHECKPOINT_PERIOD", "5",
        "TEST.IMS_PER_BATCH", "256",
        "TEST.RE_RANKING", "False",
        "TEST.TOP_K_EVAL", "0",
        "OUTPUT_DIR", str(E2_RUN_DIR),
        "PVUAD.EXPERIMENT", "E2",
        "PVUAD.STAGE", "final",
        "PVUAD.DEV_MODE", "False",
        "PVUAD.PAIRED_VIEW_SAMPLER", "True",
        "PVUAD.STRICT_PAIRED_DATA", "False",
        "PVUAD.GROUND_MAX_CAMID", str(GROUND_MAX_CAMID),
        "PVUAD.VFPROCA", "True",
        "MODEL.PCA_LOSS_WEIGHT", "0.01",
        "PVUAD.VIEW_ALIGNMENT_WEIGHT", "0.5",
        "PVUAD.INVARIANT_ID_WEIGHT", "0.5",
        "PVUAD.SCENARIO_CLS_WEIGHT", "0.1",
        "PVUAD.TEST_INVARIANT_BLEND", "0.5",
        "PVUAD.LOCAL_CONSISTENCY", "False",
        "PVUAD.SYNC_PAIRED_AUG", "False",
        "PVUAD.HARD_GPD", "False",
        "PVUAD.FINETUNE", "False",
        "PVUAD.TIR_DISTILLATION", "False",
        "PVUAD.FEATURE_PRESERVE_WEIGHT", "0.0",
        "PVUAD.SCENARIO_HARD_GPD", "False",
        "PVUAD.FINAL_FLIP_TTA", "False",
        "PVUAD.SAVE_LAST_EVERY", "5",
        "PVUAD.RUN_MANIFEST", str(manifest_path),
        "PVUAD.SESSION_DEADLINE_UNIX", str(float(SESSION_DEADLINE_UNIX)),
        "PVUAD.STOP_RESERVE_MINUTES", str(float(SESSION_STOP_RESERVE_MINUTES)),
        "PVUAD.CHECKPOINT_EVERY_MINUTES", str(float(CHECKPOINT_EVERY_MINUTES)),
        "PVUAD.TIME_CHECK_EVERY_STEPS", str(int(TIME_CHECK_EVERY_STEPS)),
        "PVUAD.MIN_EVAL_REMAINING_MINUTES", str(float(MIN_EVAL_REMAINING_MINUTES)),
        "PVUAD.STRICT_RESUME_CONFIG", str(bool(strict_resume)),
    ]
    if resume_path:
        opts += ["PVUAD.RESUME_PATH", str(resume_path)]
    return [
        sys.executable, "-m", "torch.distributed.run", "--standalone",
        f"--nproc_per_node={NUM_GPUS}", "train.py", "--config_file",
        "configs/PVUAD.yml", *opts,
    ]


E2_STATUS = SELECTED_E2["status"] if SELECTED_E2 else {}
E2_RESUME_PATH = None
E2_MODEL_PATH = None
E2_CHECKPOINT_PATH = None
E2_STRICT_RESUME = True
E2_COMPLETED_THIS_SESSION = False

if ACTIVE_STAGE == "E2":
    if SELECTED_E2 is not None:
        E2_RESUME_PATH = SELECTED_E2["checkpoint"]
        selected_hashes = (
            SELECTED_E2.get("manifest", {}).get("overlay_sha256") or {}
        )
        selected_processor_hash = selected_hashes.get("processor/processor.py")
        current_processor_hash = OVERLAY_HASHES.get("processor/processor.py")
        # E2 v2.3 checkpoints predate E3's gated fields in the resume
        # signature. Their known E2 structure is compatible; disable only the
        # signature-key comparison while retaining strict state_dict/optimizer
        # structural loading. New combined checkpoints remain fully strict.
        if (
            selected_processor_hash
            and selected_processor_hash != current_processor_hash
        ):
            E2_STRICT_RESUME = False
            print(
                "Known legacy E2 overlay detected; structural resume checks "
                "remain enabled, signature-only strictness is relaxed."
            )
        copy_if_present(
            SELECTED_E2["root"], E2_RUN_DIR,
            [
                "metrics_history.csv", "pvuad_best_model.pth", "pvuad_last_model.pth",
                "pairing_audit.csv", "ablation_summary.csv",
                "training_summary.json",
            ],
        )
    else:
        print("No checkpoint found: starting E2 from pretrained TransReID ViT weights; backbone is TRAINABLE.")

    write_pipeline_state(
        "e2_running", "E2",
        resume_path=str(E2_RESUME_PATH) if E2_RESUME_PATH else None,
    )
    E2_STATUS = run_command_checked(
        e2_command(E2_RESUME_PATH, E2_STRICT_RESUME), E2_RUN_DIR, "E2"
    )
    if not RUN_TRAINING:
        write_pipeline_state("dry_run", "E2")
    elif E2_STATUS.get("status") in {"paused_time_limit", "running_checkpoint"}:
        write_pipeline_state(
            "e2_paused", "E2", e2_status=E2_STATUS,
            next_action="Attach this output to a new session and Run All.",
        )
        print("E2 PAUSED SAFELY. Save Version and continue in a new session.")
    elif E2_STATUS.get("status") == "completed":
        E2_MODEL_PATH = E2_RUN_DIR / "pvuad_last_model.pth"
        E2_CHECKPOINT_PATH = E2_RUN_DIR / "pvuad_last_checkpoint.pth"
        required = [E2_MODEL_PATH, E2_CHECKPOINT_PATH, E2_RUN_DIR / "metrics_history.csv"]
        missing = [str(path) for path in required if not path.is_file()]
        if missing:
            raise RuntimeError(f"Completed E2 is missing: {missing}")
        E2_COMPLETED_THIS_SESSION = True
        ACTIVE_STAGE = "E3"
else:
    if SELECTED_E2 is not None:
        E2_MODEL_PATH = (
            SELECTED_E2.get("last_model")
            if SELECTED_E2.get("last_model") is not None
            and Path(SELECTED_E2["last_model"]).is_file()
            else SELECTED_E2["model"]
        )
        E2_CHECKPOINT_PATH = SELECTED_E2["checkpoint"]


def create_teacher_cache(source_model, destination):
    source_model = Path(source_model)
    destination = Path(destination)
    if destination.is_file():
        return destination
    if not source_model.is_file():
        raise FileNotFoundError(source_model)
    try:
        payload = torch.load(source_model, map_location="cpu", weights_only=False)
    except TypeError:
        payload = torch.load(source_model, map_location="cpu")
    if isinstance(payload, dict) and "model" in payload:
        payload = payload["model"]
    elif isinstance(payload, dict) and "state_dict" in payload:
        payload = payload["state_dict"]
    if not isinstance(payload, dict):
        raise ValueError("E2 model file does not contain a state dictionary")
    teacher_state = {}
    for key, value in payload.items():
        clean = str(key).replace("module.", "")
        teacher_state[clean] = value.half() if torch.is_floating_point(value) else value
    temporary = destination.with_suffix(".pth.tmp")
    torch.save(teacher_state, temporary)
    os.replace(temporary, destination)
    del payload, teacher_state
    return destination


E3_STATUS = SELECTED_E3["status"] if SELECTED_E3 else {}
E3_RESUME_PATH = SELECTED_E3["checkpoint"] if SELECTED_E3 else None
TEACHER_CACHE = E3_RUN_DIR / "e2_teacher_model_fp16.pth"

if SELECTED_E3 is not None:
    copy_if_present(
        SELECTED_E3["root"], E3_RUN_DIR,
        [
            "metrics_history.csv", "pvuad_best_model.pth", "pvuad_last_model.pth",
            "e2_teacher_model_fp16.pth", "tta_best_metrics.csv", "tta_last_metrics.csv",
            "training_summary.json", "TRAINING_COMPLETE_BEFORE_TTA.json",
            "FINAL_TTA_FAILED.txt", "effective_test_command.txt",
        ],
    )


def e3_command(resume_path, student_model, prototype_checkpoint, teacher_cache):
    recipe = {
        "experiment": "E3-TDH", "stage": "final", "epochs": E3_EPOCHS,
        "global_batch": E3_GLOBAL_BATCH, "backbone_lr": E3_BASE_LR,
        "head_lr_multiplier": E3_HEAD_LR_MULTIPLIER,
        "tir_global_weight": 0.0,
        "tir_local_weight": 0.0,
        "feature_preserve_weight": FEATURE_PRESERVE_WEIGHT,
        "scenario_hard_weight": SCENARIO_HARD_WEIGHT,
        "initialization_origin": "MSMT17_TransReID_then_AGReIDv2_E2_epoch60",
        "dataset": "AG-ReID.v2", "protocol": "aerial_to_ground",
        "tir_distillation": False,
        "embedding_dim": 768,
        "backbone_stride": [16, 16],
    }
    manifest_path = make_stage_manifest(
        E3_RUN_DIR, "E3-TDH", recipe, resume_path=resume_path,
        e2_model_path=str(E2_MODEL_PATH) if E2_MODEL_PATH else None,
        e2_checkpoint_path=str(E2_CHECKPOINT_PATH) if E2_CHECKPOINT_PATH else None,
        teacher_model_path=str(teacher_cache),
    )
    opts = [
        "MODEL.DIST_TRAIN", "True",
        "MODEL.PRETRAIN_CHOICE", "none",
        "MODEL.PRETRAIN_PATH", str(PRETRAIN_PATH),
        "MODEL.STRIDE_SIZE", "[16, 16]",
        "MODEL.SIE_CAMERA", "False",
        "MODEL.SIE_VIEW", "False",
        "DATASETS.NAMES", "AG-ReID.v2",
        "DATASETS.MODALITIES", "['RGB']",
        "DATASETS.ROOT_DIR", str(DATA_LINK_PARENT),
        "DATALOADER.NUM_WORKERS", str(NUM_WORKERS),
        "SOLVER.SEED", str(RANDOM_SEED),
        "SOLVER.MAX_EPOCHS", str(E3_EPOCHS),
        "SOLVER.IMS_PER_BATCH", str(E3_GLOBAL_BATCH),
        "SOLVER.BASE_LR", str(E3_BASE_LR),
        "SOLVER.WARMUP_EPOCHS", "1",
        "SOLVER.EVAL_PERIOD", "5",
        "SOLVER.CHECKPOINT_PERIOD", "5",
        "INPUT.RE_PROB", "0.0",
        "TEST.IMS_PER_BATCH", "256",
        "TEST.RE_RANKING", "False",
        "TEST.TOP_K_EVAL", "0",
        "OUTPUT_DIR", str(E3_RUN_DIR),
        "PVUAD.EXPERIMENT", "E3-TDH",
        "PVUAD.STAGE", "final",
        "PVUAD.DEV_MODE", "False",
        "PVUAD.PAIRED_VIEW_SAMPLER", "True",
        "PVUAD.STRICT_PAIRED_DATA", "False",
        "PVUAD.GROUND_MAX_CAMID", str(GROUND_MAX_CAMID),
        "PVUAD.SYNC_PAIRED_AUG", "True",
        "PVUAD.VFPROCA", "True",
        "MODEL.PCA_LOSS_WEIGHT", "0.01",
        "PVUAD.VIEW_ALIGNMENT_WEIGHT", "0.5",
        "PVUAD.INVARIANT_ID_WEIGHT", "0.5",
        "PVUAD.SCENARIO_CLS_WEIGHT", "0.1",
        "PVUAD.TEST_INVARIANT_BLEND", "0.5",
        "PVUAD.LOCAL_CONSISTENCY", "False",
        "PVUAD.HARD_GPD", "False",
        "PVUAD.FINETUNE", "True",
        "PVUAD.WARMSTART_MODEL_PATH", str(student_model),
        "PVUAD.WARMSTART_PROTOTYPE_PATH", (
            str(prototype_checkpoint) if prototype_checkpoint else ""
        ),
        "PVUAD.TEACHER_MODEL_PATH", str(teacher_cache),
        "PVUAD.HEAD_LR_MULTIPLIER", str(float(E3_HEAD_LR_MULTIPLIER)),
        "PVUAD.TIR_DISTILLATION", "False",
        "PVUAD.TIR_GLOBAL_WEIGHT", "0.0",
        "PVUAD.TIR_LOCAL_WEIGHT", "0.0",
        "PVUAD.FEATURE_PRESERVE_WEIGHT", str(float(FEATURE_PRESERVE_WEIGHT)),
        "PVUAD.SCENARIO_HARD_GPD", "True",
        "PVUAD.SCENARIO_HARD_WEIGHT", str(float(SCENARIO_HARD_WEIGHT)),
        "PVUAD.SCENARIO_HARD_TOPK", str(int(SCENARIO_HARD_TOPK)),
        "PVUAD.SCENARIO_HARD_MARGIN", str(float(SCENARIO_HARD_MARGIN)),
        "PVUAD.SCENARIO_HARD_TAU", str(float(SCENARIO_HARD_TAU)),
        "PVUAD.SCENARIO_HARD_START_EPOCH", str(int(SCENARIO_HARD_START_EPOCH)),
        "PVUAD.SCENARIO_BANK_MOMENTUM", "0.2",
        "PVUAD.FINAL_FLIP_TTA", "False",
        "PVUAD.SAVE_LAST_EVERY", "5",
        "PVUAD.RUN_MANIFEST", str(manifest_path),
        "PVUAD.SESSION_DEADLINE_UNIX", str(float(SESSION_DEADLINE_UNIX)),
        "PVUAD.STOP_RESERVE_MINUTES", str(float(SESSION_STOP_RESERVE_MINUTES)),
        "PVUAD.CHECKPOINT_EVERY_MINUTES", str(float(CHECKPOINT_EVERY_MINUTES)),
        "PVUAD.TIME_CHECK_EVERY_STEPS", str(int(TIME_CHECK_EVERY_STEPS)),
        "PVUAD.MIN_EVAL_REMAINING_MINUTES", str(float(MIN_EVAL_REMAINING_MINUTES)),
        "PVUAD.STRICT_RESUME_CONFIG", "True",
    ]
    if resume_path:
        opts += ["PVUAD.RESUME_PATH", str(resume_path)]
    return [
        sys.executable, "-m", "torch.distributed.run", "--standalone",
        f"--nproc_per_node={NUM_GPUS}", "train.py", "--config_file",
        "configs/PVUAD.yml", *opts,
    ]


can_enter_e3 = (
    ACTIVE_STAGE == "E3"
    and (E3_STATUS.get("status") != "completed")
    and RUN_TRAINING
)
if (
    can_enter_e3
    and E2_COMPLETED_THIS_SESSION
    and ALWAYS_START_E3_IN_NEW_SESSION
):
    write_pipeline_state(
        "e2_completed_e3_pending", "E3",
        e2_status=E2_STATUS,
        remaining_minutes=remaining_minutes(),
        next_action=(
            "Kaggle will save this completed E2 output. Attach it to a new "
            "session and use Save & Run All; E3 will start automatically."
        ),
    )
    print(
        "E2 COMPLETE. This Kaggle version now ends cleanly by design. "
        "Attach its output to the next session; E3 will start there."
    )
    can_enter_e3 = False
elif can_enter_e3 and remaining_minutes() < MIN_E3_START_REMAINING_MINUTES:
    pending_status = "e3_paused" if E3_RESUME_PATH else "e2_completed_e3_pending"
    write_pipeline_state(
        pending_status, "E3",
        e2_status=E2_STATUS,
        remaining_minutes=remaining_minutes(),
        next_action="Save Version; attach this output and Run All in a new session.",
    )
    print(
        "Only {:.1f} minutes remain. E3 will start/resume in the next "
        "session.".format(remaining_minutes())
    )
    can_enter_e3 = False

if can_enter_e3:
    if E3_RESUME_PATH is not None:
        teacher_source = (
            SELECTED_E3["root"] / "e2_teacher_model_fp16.pth"
        )
        if not teacher_source.is_file() and E2_MODEL_PATH is not None:
            teacher_source = E2_MODEL_PATH
        if not TEACHER_CACHE.is_file():
            if teacher_source.name == "e2_teacher_model_fp16.pth":
                shutil.copy2(teacher_source, TEACHER_CACHE)
            else:
                create_teacher_cache(teacher_source, TEACHER_CACHE)
        student_model = TEACHER_CACHE
        prototype_checkpoint = None
    else:
        if E2_MODEL_PATH is None or not Path(E2_MODEL_PATH).is_file():
            raise FileNotFoundError(
                "E3 needs a completed E2 final model. Attach the "
                "previous combined/E2 output or use E2_COMPLETED_DIR_OVERRIDE."
            )
        if E2_CHECKPOINT_PATH is None or not Path(E2_CHECKPOINT_PATH).is_file():
            raise FileNotFoundError(
                "E3 needs E2 pvuad_last_checkpoint.pth for prototype warm-start."
            )
        create_teacher_cache(E2_MODEL_PATH, TEACHER_CACHE)
        student_model = E2_MODEL_PATH
        prototype_checkpoint = E2_CHECKPOINT_PATH

    print("Frozen E2 teacher:", TEACHER_CACHE)
    write_pipeline_state(
        "e3_running", "E3",
        e2_status=E2_STATUS,
        e3_resume_path=str(E3_RESUME_PATH) if E3_RESUME_PATH else None,
    )
    E3_STATUS = run_command_checked(
        e3_command(
            E3_RESUME_PATH, student_model, prototype_checkpoint, TEACHER_CACHE
        ),
        E3_RUN_DIR,
        "E3-TDH (AG adaptation)",
    )
    if E3_STATUS.get("status") in {"paused_time_limit", "running_checkpoint"}:
        write_pipeline_state(
            "e3_paused", "E3", e2_status=E2_STATUS, e3_status=E3_STATUS,
            next_action="Attach this output to a new session and Run All.",
        )
        print("E3 PAUSED SAFELY. Save Version and continue in a new session.")


def run_final_tta(model_path, metrics_filename, label):
    model_path = Path(model_path)
    metrics_path = E3_RUN_DIR / metrics_filename
    if metrics_path.is_file():
        print(f"{label}: already evaluated -> {metrics_path}")
        return True

    test_opts = [
        "MODEL.DIST_TRAIN", "False",
        "MODEL.PRETRAIN_CHOICE", "none",
        "MODEL.PRETRAIN_PATH", str(PRETRAIN_PATH),
        "MODEL.STRIDE_SIZE", "[16, 16]",
        "MODEL.SIE_CAMERA", "False",
        "MODEL.SIE_VIEW", "False",
        "DATASETS.NAMES", "AG-ReID.v2",
        "DATASETS.MODALITIES", "['RGB']",
        "DATASETS.ROOT_DIR", str(DATA_LINK_PARENT),
        "DATALOADER.NUM_WORKERS", str(NUM_WORKERS),
        "TEST.IMS_PER_BATCH", "256",
        "TEST.RE_RANKING", "False",
        "TEST.TOP_K_EVAL", "0",
        "TEST.WEIGHT", str(model_path),
        "OUTPUT_DIR", str(E3_RUN_DIR),
        "PVUAD.EXPERIMENT", "E3-TDH",
        "PVUAD.STAGE", "final",
        "PVUAD.DEV_MODE", "False",
        "PVUAD.PAIRED_VIEW_SAMPLER", "True",
        "PVUAD.STRICT_PAIRED_DATA", "False",
        "PVUAD.GROUND_MAX_CAMID", str(GROUND_MAX_CAMID),
        "PVUAD.VFPROCA", "True",
        "PVUAD.TEST_INVARIANT_BLEND", "0.5",
        "PVUAD.TIR_DISTILLATION", "False",
        "PVUAD.FEATURE_PRESERVE_WEIGHT", "0.0",
        "PVUAD.SCENARIO_HARD_GPD", "True",
        "PVUAD.FINAL_FLIP_TTA", "True",
        "PVUAD.TTA_METRICS_FILENAME", metrics_filename,
    ]
    test_command = [
        sys.executable, "test.py", "--config_file", "configs/PVUAD.yml",
        *test_opts,
    ]
    safe_label = re.sub(r"[^A-Za-z0-9_-]+", "_", label)
    (E3_RUN_DIR / f"effective_test_command_{safe_label}.txt").write_text(
        shlex.join(test_command) + "\n", encoding="utf-8"
    )
    print(f"Running AG-ReID.v2 A2G flip-TTA: {label}")
    result = subprocess.run(
        test_command, cwd=UAD_DIR, env=environment, check=False
    )
    if result.returncode != 0 or not metrics_path.is_file():
        warning = (
            f"{label} TTA failed after E3 training; checkpoint remains valid. "
            f"returncode={result.returncode}."
        )
        (E3_RUN_DIR / f"FINAL_TTA_FAILED_{safe_label}.txt").write_text(
            warning + "\n", encoding="utf-8"
        )
        print("WARNING:", warning)
        return False
    return True


E3_COMPLETED = E3_STATUS.get("status") == "completed"
if E3_COMPLETED:
    if SELECTED_E3 is not None:
        copy_if_present(
            SELECTED_E3["root"], E3_RUN_DIR,
            [
                "pvuad_last_checkpoint.pth", "pvuad_last_model.pth",
                "pvuad_best_model.pth", "session_status.json",
                "training_summary.json", "metrics_history.csv",
                "tta_best_metrics.csv", "tta_last_metrics.csv",
            ],
        )

    best_model = E3_RUN_DIR / "pvuad_best_model.pth"
    last_model = E3_RUN_DIR / "pvuad_last_model.pth"
    if not best_model.is_file():
        raise FileNotFoundError("Completed E3 is missing pvuad_best_model.pth")
    if not last_model.is_file():
        raise FileNotFoundError("Completed E3 is missing pvuad_last_model.pth")

    best_tta_path = E3_RUN_DIR / "tta_best_metrics.csv"
    last_tta_path = E3_RUN_DIR / "tta_last_metrics.csv"

    best_tta_ok = best_tta_path.is_file()
    last_tta_ok = last_tta_path.is_file()

    if RUN_FINAL_FLIP_TTA and not best_tta_ok:
        if remaining_minutes() >= MIN_TTA_START_REMAINING_MINUTES:
            best_tta_ok = run_final_tta(
                best_model, "tta_best_metrics.csv", "best_checkpoint"
            )
        else:
            print(
                "E3 complete but insufficient time for best-checkpoint TTA; "
                "continue next session."
            )

    # If the best checkpoint is the requested final epoch 25, best and last
    # are the same training state. Reuse the already-computed TTA metrics.
    e3_summary = read_json(E3_RUN_DIR / "training_summary.json")
    best_epoch = int(e3_summary.get("best_epoch") or 0)
    if (
        RUN_FINAL_FLIP_TTA
        and not last_tta_ok
        and best_epoch == E3_EPOCHS
        and best_tta_path.is_file()
    ):
        shutil.copy2(best_tta_path, last_tta_path)
        last_tta_ok = True
        print(
            f"best_epoch == final epoch == {E3_EPOCHS}; "
            "best and last checkpoint are the same training state. "
            "Copied TTA metrics instead of repeating inference."
        )

    if RUN_FINAL_FLIP_TTA and not last_tta_ok:
        if remaining_minutes() >= MIN_TTA_START_REMAINING_MINUTES:
            last_tta_ok = run_final_tta(
                last_model, "tta_last_metrics.csv", "last_epoch25"
            )
        else:
            print(
                "E3 complete but insufficient time for last-epoch TTA; "
                "continue next session."
            )

    tta_ok = (
        (best_tta_ok and last_tta_ok)
        if RUN_FINAL_FLIP_TTA else True
    )
    final_status = "completed" if tta_ok else "e3_complete_tta_pending"
    write_pipeline_state(
        final_status,
        "DONE" if final_status == "completed" else "TTA",
        e2_status=E2_STATUS,
        e3_status=E3_STATUS,
        best_epoch=best_epoch or None,
        tta_best_complete=bool(best_tta_ok),
        tta_last_complete=bool(last_tta_ok),
        next_action=(
            None if final_status == "completed"
            else "Attach this output and Run All; training is skipped and pending TTA resumes."
        ),
    )
elif ACTIVE_STAGE == "E3" and not can_enter_e3 and not RUN_TRAINING:
    write_pipeline_state("dry_run", "E3")
elif ACTIVE_STAGE == "E3" and not can_enter_e3 and not RUN_TRAINING:
    write_pipeline_state("dry_run", "E3")


Frozen E2 teacher: /kaggle/working/pvuad_runs/agreidv2_a2g_e2_60_e3_25_transreid768_b24_v5_manifestfix/e3_tdh/e2_teacher_model_fp16.pth

Launching E3-TDH (AG adaptation)
/usr/bin/python3 -m torch.distributed.run --standalone --nproc_per_node=2 train.py --config_file configs/PVUAD.yml MODEL.DIST_TRAIN True MODEL.PRETRAIN_CHOICE none MODEL.PRETRAIN_PATH /kaggle/working/vit_transreid_msmt.pth MODEL.STRIDE_SIZE '[16, 16]' MODEL.SIE_CAMERA False MODEL.SIE_VIEW False DATASETS.NAMES AG-ReID.v2 DATASETS.MODALITIES '['"'"'RGB'"'"']' DATASETS.ROOT_DIR /kaggle/working/pvuad_agreid_data DATALOADER.NUM_WORKERS 4 SOLVER.SEED 1234 SOLVER.MAX_EPOCHS 25 SOLVER.IMS_PER_BATCH 24 SOLVER.BASE_LR 0.0001 SOLVER.WARMUP_EPOCHS 1 SOLVER.EVAL_PERIOD 5 SOLVER.CHECKPOINT_PERIOD 5 INPUT.RE_PROB 0.0 TEST.IMS_PER_BATCH 256 TEST.RE_RANKING False TEST.TOP_K_EVAL 0 OUTPUT_DIR /kaggle/working/pvuad_runs/agreidv2_a2g_e2_60_e3_25_transreid768_b24_v5_manifestfix/e3_tdh PVUAD.EXPERIMENT E3-TDH PVUAD.STAGE final PVUAD.DEV_MOD

[W914 04:02:09.886074992 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3


2026-09-14 04:02:23,121 transreid INFO: Saving model in the path :/kaggle/working/pvuad_runs/agreidv2_a2g_e2_60_e3_25_transreid768_b24_v5_manifestfix/e3_tdh
2026-09-14 04:02:23,122 transreid INFO: Namespace(config_file='configs/PVUAD.yml', opts=['MODEL.DIST_TRAIN', 'True', 'MODEL.PRETRAIN_CHOICE', 'none', 'MODEL.PRETRAIN_PATH', '/kaggle/working/vit_transreid_msmt.pth', 'MODEL.STRIDE_SIZE', '[16, 16]', 'MODEL.SIE_CAMERA', 'False', 'MODEL.SIE_VIEW', 'False', 'DATASETS.NAMES', 'AG-ReID.v2', 'DATASETS.MODALITIES', "['RGB']", 'DATASETS.ROOT_DIR', '/kaggle/working/pvuad_agreid_data', 'DATALOADER.NUM_WORKERS', '4', 'SOLVER.SEED', '1234', 'SOLVER.MAX_EPOCHS', '25', 'SOLVER.IMS_PER_BATCH', '24', 'SOLVER.BASE_LR', '0.0001', 'SOLVER.WARMUP_EPOCHS', '1', 'SOLVER.EVAL_PERIOD', '5', 'SOLVER.CHECKPOINT_PERIOD', '5', 'INPUT.RE_PROB', '0.0', 'TEST.IMS_PER_BATCH', '256', 'TEST.RE_RANKING', 'False', 'TEST.TOP_K_EVAL', '0', 'OUTPUT_DIR', '/kaggle/working/pvuad_runs/agreidv2_a2g_e2_60_e3_25_transreid768_b2

[W914 04:02:23.413111646 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[W914 04:02:23.420626125 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3


=> AG-ReID.v2 loaded (aerial -> ground)
   train: /kaggle/working/pvuad_agreid_data/AG-ReID.v2/train_all
   query: /kaggle/working/pvuad_agreid_data/AG-ReID.v2/query
   gallery: ['/kaggle/working/pvuad_agreid_data/AG-ReID.v2/gallery']
=> AG-ReID.v2 loaded (aerial -> ground)
   train: /kaggle/working/pvuad_agreid_data/AG-ReID.v2/train_all
   query: /kaggle/working/pvuad_agreid_data/AG-ReID.v2/query
   gallery: ['/kaggle/working/pvuad_agreid_data/AG-ReID.v2/gallery']
Dataset statistics:
  ----------------------------------------
  subset   | # ids | # images | # cameras
  ----------------------------------------
  train    |   807 |    51530 |         3
  query    |   808 |     4348 |         1
  gallery  |   808 |    19259 |         2
  ----------------------------------------
Dataset statistics:
  ----------------------------------------
  subset   | # ids | # images | # cameras
  ----------------------------------------
  train    |   807 |    51530 |         3
  query    |   808 |   

/usr/local/lib/python3.12/dist-packages/torch/distributed/c10d_logger.py:83: UserWarning: barrier(): using the device under current context. You can specify `device_id` in `init_process_group` to mute this warning.
  return func(*args, **kwargs)


2026-09-14 04:35:50,241 transreid.train INFO: Epoch[3] Iteration[3100/3102] Loss 0.436 Acc 0.985 ID 0.205 TRI 0.016 Align 0.040 GPD 0.008 TIR-G 0.000 TIR-L 0.000 Keep 0.030 S-Hard 0.002 LR 9.65e-05
2026-09-14 04:35:50,241 transreid.train INFO: Epoch[3] Iteration[3100/3102] Loss 0.443 Acc 0.984 ID 0.209 TRI 0.016 Align 0.040 GPD 0.007 TIR-G 0.000 TIR-L 0.000 Keep 0.030 S-Hard 0.002 LR 9.65e-05
2026-09-14 04:35:50,705 transreid.train INFO: Epoch 3 done in 617.7s; loss 0.4356; acc 0.9847
2026-09-14 04:36:02,381 transreid.train INFO: Epoch[4] Iteration[50/3102] Loss 0.948 Acc 0.938 ID 0.455 TRI 0.021 Align 0.044 GPD 0.017 TIR-G 0.000 TIR-L 0.000 Keep 0.035 S-Hard 0.004 LR 9.38e-05
2026-09-14 04:36:02,381 transreid.train INFO: Epoch[4] Iteration[50/3102] Loss 0.922 Acc 0.947 ID 0.465 TRI 0.017 Align 0.045 GPD 0.017 TIR-G 0.000 TIR-L 0.000 Keep 0.036 S-Hard 0.003 LR 9.38e-05
2026-09-14 04:36:12,251 transreid.train INFO: Epoch[4] Iteration[100/3102] Loss 0.898 Acc 0.945 ID 0.444 TRI 0.017 Ali

,stage,status,reason,epoch,next_iteration,epoch_complete,global_step,best_mAP,remaining_minutes
0,E3-TDH (AG adaptation),completed,training_completed,25,0,True,77574,73.354912,326.8


Running AG-ReID.v2 A2G flip-TTA: best_checkpoint
2026-09-14 08:45:36,964 transreid INFO: Namespace(config_file='configs/PVUAD.yml', opts=['MODEL.DIST_TRAIN', 'False', 'MODEL.PRETRAIN_CHOICE', 'none', 'MODEL.PRETRAIN_PATH', '/kaggle/working/vit_transreid_msmt.pth', 'MODEL.STRIDE_SIZE', '[16, 16]', 'MODEL.SIE_CAMERA', 'False', 'MODEL.SIE_VIEW', 'False', 'DATASETS.NAMES', 'AG-ReID.v2', 'DATASETS.MODALITIES', "['RGB']", 'DATASETS.ROOT_DIR', '/kaggle/working/pvuad_agreid_data', 'DATALOADER.NUM_WORKERS', '4', 'TEST.IMS_PER_BATCH', '256', 'TEST.RE_RANKING', 'False', 'TEST.TOP_K_EVAL', '0', 'TEST.WEIGHT', '/kaggle/working/pvuad_runs/agreidv2_a2g_e2_60_e3_25_transreid768_b24_v5_manifestfix/e3_tdh/pvuad_best_model.pth', 'OUTPUT_DIR', '/kaggle/working/pvuad_runs/agreidv2_a2g_e2_60_e3_25_transreid768_b24_v5_manifestfix/e3_tdh', 'PVUAD.EXPERIMENT', 'E3-TDH', 'PVUAD.STAGE', 'final', 'PVUAD.DEV_MODE', 'False', 'PVUAD.PAIRED_VIEW_SAMPLER', 'True', 'PVUAD.STRICT_PAIRED_DATA', 'False', 'PVUAD.GROUND_MAX

## 6. Tổng hợp kết quả

Report cuối chứa:

- E2 best và E2 epoch 60;
- E3 best raw và E3 epoch 25 raw;
- E3 best + flip-TTA;
- E3 epoch 25 + flip-TTA;
- dataset/layout audit;
- `checkpoint_relation.json` để xác nhận best epoch có bằng last epoch không.

File report:
`/kaggle/working/pvuad_agreidv2_a2g_e2_60_e3_25_transreid_reports.zip`

In [7]:
def metrics_history_row(run_dir, epoch=None, best=False):
    path = Path(run_dir) / "metrics_history.csv"
    if not path.is_file():
        return None
    frame = pd.read_csv(path)
    required = {"experiment", "stage", "epoch", "mAP", "Rank-1", "Rank-5", "Rank-10"}
    if frame.empty or not required.issubset(frame.columns):
        return None
    frame = frame.drop_duplicates(
        subset=["experiment", "stage", "epoch"], keep="last"
    ).sort_values("epoch")
    frame.to_csv(path, index=False)
    if best:
        row = frame.loc[frame["mAP"].idxmax()]
    elif epoch is not None:
        matched = frame.loc[frame["epoch"].astype(int) == int(epoch)]
        if matched.empty:
            return None
        row = matched.iloc[-1]
    else:
        row = frame.iloc[-1]
    return {
        "epoch": int(row["epoch"]),
        "mAP": float(row["mAP"]),
        "Rank-1": float(row["Rank-1"]),
        "Rank-5": float(row["Rank-5"]),
        "Rank-10": float(row["Rank-10"]),
    }

def tta_metrics(path, label, epoch_label):
    path = Path(path)
    if not path.is_file():
        return None
    frame = pd.read_csv(path)
    if frame.empty:
        return None
    row = frame.iloc[-1]
    return {
        "method": label,
        "epoch": epoch_label,
        "mAP": float(row["mAP"]),
        "Rank-1": float(row["Rank-1"]),
        "Rank-5": float(row["Rank-5"]),
        "Rank-10": float(row["Rank-10"]),
    }

state = read_json(PIPELINE_STATE_PATH)
print("Pipeline status:", state.get("status"))
print("Active stage   :", state.get("active_stage"))
print("Next action    :", state.get("next_action"))
print("Dataset        :", DATASET_AUDIT)

init_audit_path = PIPELINE_ROOT / "transreid_trainable_init_audit.json"
if init_audit_path.is_file():
    print("TransReID initialization audit:")
    display(pd.DataFrame([read_json(init_audit_path)]))

comparison = []

e2_best = metrics_history_row(E2_RUN_DIR, best=True)
e2_last = metrics_history_row(E2_RUN_DIR, epoch=E2_EPOCHS)
if e2_best:
    comparison.append({
        "method": "E2 best checkpoint (diagnostic)",
        **e2_best,
    })
if e2_last:
    comparison.append({
        "method": f"E2 fixed final epoch {E2_EPOCHS}",
        **e2_last,
    })

e3_best = metrics_history_row(E3_RUN_DIR, best=True)
e3_last = metrics_history_row(E3_RUN_DIR, epoch=E3_EPOCHS)
if e3_best:
    comparison.append({
        "method": "E3 best checkpoint (no TTA)",
        **e3_best,
    })
if e3_last:
    comparison.append({
        "method": f"E3 fixed final epoch {E3_EPOCHS} (no TTA)",
        **e3_last,
    })

best_tta = tta_metrics(
    E3_RUN_DIR / "tta_best_metrics.csv",
    "E3 best checkpoint + horizontal-flip TTA",
    e3_best["epoch"] if e3_best else "best",
)
last_tta = tta_metrics(
    E3_RUN_DIR / "tta_last_metrics.csv",
    f"E3 fixed final epoch {E3_EPOCHS} + horizontal-flip TTA",
    E3_EPOCHS,
)
if best_tta:
    comparison.append(best_tta)
if last_tta:
    comparison.append(last_tta)

comparison_frame = pd.DataFrame(comparison)
comparison_frame.to_csv(
    PIPELINE_ROOT / "combined_results.csv", index=False
)
if not comparison_frame.empty:
    display(comparison_frame.round(4))

best_epoch = int(e3_best["epoch"]) if e3_best else None
best_equals_last = bool(best_epoch == E3_EPOCHS) if best_epoch is not None else None
checkpoint_relation = {
    "E2_fixed_horizon": E2_EPOCHS,
    "E3_fixed_horizon": E3_EPOCHS,
    "E3_best_epoch": best_epoch,
    "E3_best_epoch_equals_last_epoch": best_equals_last,
    "primary_protocol": "AG-ReID.v2 aerial_to_ground",
    "note": (
        "E3 is warm-started from the final E2 epoch-60 model and the same "
        "epoch-60 prototype checkpoint. Both best-E3 and fixed epoch-25 "
        "flip-TTA metrics are reported so best-vs-last is empirical, not assumed."
    ),
}
(PIPELINE_ROOT / "checkpoint_relation.json").write_text(
    json.dumps(checkpoint_relation, indent=2), encoding="utf-8"
)
print("Checkpoint relation:", checkpoint_relation)

REPORT_ZIP = (
    Path("/kaggle/working")
    / "pvuad_agreidv2_a2g_e2_60_e3_25_transreid_reports.zip"
)
if REPORT_ZIP.exists():
    REPORT_ZIP.unlink()

report_suffixes = {".csv", ".json", ".txt", ".log", ".yml", ".yaml"}
with zipfile.ZipFile(
    REPORT_ZIP, "w", compression=zipfile.ZIP_DEFLATED
) as archive:
    for path in sorted(PIPELINE_ROOT.rglob("*")):
        if path.is_file() and path.suffix.lower() in report_suffixes:
            archive.write(
                path,
                arcname=str(path.relative_to(PIPELINE_ROOT)),
            )
    dataset_audit_path = Path("/kaggle/working/agreid_v2_dataset_audit.json")
    if dataset_audit_path.is_file():
        archive.write(dataset_audit_path, arcname="agreid_v2_dataset_audit.json")
    patch_manifest = UAD_DIR / "PVUAD_patch_manifest.json"
    if patch_manifest.is_file():
        archive.write(patch_manifest, arcname="PVUAD_patch_manifest.json")

print("Report zip:", REPORT_ZIP)

Pipeline status: completed
Active stage   : DONE
Next action    : None
Dataset        : {'dataset': 'AG-ReID.v2', 'protocol': 'aerial_to_ground', 'root': '/kaggle/input/datasets/thienbao1604/ag-reid-v2/AG-ReID.v2/AG-ReID.v2', 'train_dir': '/kaggle/input/datasets/thienbao1604/ag-reid-v2/AG-ReID.v2/AG-ReID.v2/train_all', 'query_dir': '/kaggle/input/datasets/thienbao1604/ag-reid-v2/AG-ReID.v2/AG-ReID.v2/query', 'gallery_dirs': ['/kaggle/input/datasets/thienbao1604/ag-reid-v2/AG-ReID.v2/AG-ReID.v2/gallery'], 'train_images': 51530, 'train_ids': 807, 'query_images': 4348, 'query_ids': 808, 'gallery_images': 19259, 'gallery_ids': 808, 'camera_remap': {'C2_ground': 0, 'C3_ground': 1, 'C0_aerial': 2}, 'recursive_split_scan': True}
TransReID initialization audit:


,checkpoint,loaded_tensor_count,target_tensor_count,mismatch_count,resized_pos_embed,source_sie_loaded,source_jpm_local_branches_loaded,global_jpm_branch_used_for_target_final_block,global_branch_mapped_tensor_count,embedding_dim,...,batchnorm_train_mode_checked,init_qkv_max_abs,global_final_qkv_max_abs,qkv_grad_norm,patch_grad_norm,backbone_all_trainable,source_sie_used,source_jpm_local_branches_used,source_jpm_global_final_branch_used,target_stride
0,/kaggle/working/vit_transreid_msmt.pth,152,152,0,"{'source_shape': [1, 211, 768], 'target_shape'...",False,False,True,14,768,...,True,0.0,0.0,0.006533,0.032783,True,False,False,True,"[16, 16]"


,method,epoch,mAP,Rank-1,Rank-5,Rank-10
0,E2 best checkpoint (diagnostic),60,72.6778,84.3376,87.9715,89.6734
1,E2 fixed final epoch 60,60,72.6778,84.3376,87.9715,89.6734
2,E3 best checkpoint (no TTA),25,73.3549,84.6136,88.2475,89.8574
3,E3 fixed final epoch 25 (no TTA),25,73.3549,84.6136,88.2475,89.8574
4,E3 best checkpoint + horizontal-flip TTA,25,73.7924,84.4986,89.8574,92.0423
5,E3 fixed final epoch 25 + horizontal-flip TTA,25,73.7924,84.4986,89.8574,92.0423


Checkpoint relation: {'E2_fixed_horizon': 60, 'E3_fixed_horizon': 25, 'E3_best_epoch': 25, 'E3_best_epoch_equals_last_epoch': True, 'primary_protocol': 'AG-ReID.v2 aerial_to_ground', 'note': 'E3 is warm-started from the final E2 epoch-60 model and the same epoch-60 prototype checkpoint. Both best-E3 and fixed epoch-25 flip-TTA metrics are reported so best-vs-last is empirical, not assumed.'}
Report zip: /kaggle/working/pvuad_agreidv2_a2g_e2_60_e3_25_transreid_reports.zip


## Cách chạy trên Kaggle

1. Chọn **GPU T4 ×2**.
2. Attach dataset:
   `/kaggle/input/datasets/thienbao1604/ag-reid-v2/AG-ReID.v2/AG-ReID.v2.`
3. Phiên đầu: bật Internet để tải official `vit_transreid_msmt.pth`,
   hoặc attach checkpoint đó làm Kaggle Input.
4. Chọn **Save Version → Save & Run All**.
5. Nếu notebook dừng an toàn vì giới hạn phiên, ở phiên sau attach **output của
   chính version ngay trước đó** làm Input rồi chạy lại **Save & Run All**.
6. E2 hoàn tất ở epoch 60 thì notebook chủ động kết thúc version; phiên sau mới
   bắt đầu E3.
7. Sau E3 epoch 25, notebook chạy flip-TTA. Nếu best epoch = 25 thì best/last là
   cùng training state và metrics được reuse; nếu best epoch khác 25, notebook
   evaluate cả hai checkpoint.
8. Khi xong gửi lại:
   `pvuad_agreidv2_a2g_e2_60_e3_25_transreid_reports.zip`
   để mình đọc kết quả và so với các baseline AG-ReID.v2.